# PKG reference data (v2)

**PNC Treasury Management · DSI · Payment Knowledge Graph**

Builds the lookup tables that the counterparty and customer metrics join to. Run it after `pkg_edges_refresh.ipynb` and
before the metrics notebooks. Every run rebuilds every table from all months of the edge tables.

| Table | Grain | Was built by |
|---|---|---|
| `pkg_cpty_dim` | one row per counterparty id | `pkg_edges_2024_backfill.ipynb` §5 (same logic) |
| `pkg_cpty_entity_map_v2` | one row per counterparty id: `ent_l0`, `ent_l1`, `ent_l2`, `ent_l2_fi`, rule, name class | `pkg_cpty_entity_map_v2.ipynb` |
| `pkg_fi_routing` | one row per 9-digit routing number: canonical bank name | `pkg_fi_canonical.ipynb` |
| `pkg_fi_name_map` | raw bank name → canonical bank name | `pkg_fi_canonical.ipynb` |
| `pkg_cust_dim` | one row per `mdm_id` | `pkg_customer_metrics_v3.ipynb` (same logic) |
| `pkg_cpty_replacement` | one row per replaced counterparty: customer × direction × old entity → new entity | new in v2 |

**Changes from the earlier notebooks**
- `ent_l1` (6C P2P ids folded into their 5C id) is built here, so the v1 map `pkg_cpty_entity_map` is no longer needed
  and no longer updated.
- **One row per counterparty id in the map:** if an id ever has more than one row, the lowest `ent_l2` is kept and §3
  reports it.
- **Name keys ignore spaces** (`KEY_IGNORE_SPACES`): HARLEYDAVIDSON and HARLEY DAVIDSON are one merge key, so the brand
  merges across banks when any of its spellings is distinctive.
- **Bank-name rules** (`FI_FIX_RULES`): every raw bank name containing VETERANS AFFAIRS takes the name of the US
  Treasury found in the data, so federal payments (VACP TREAS 310) no longer flip between two banks. §4 shows which names
  were matched.
- The review sections (typer eyeball tables, v1 → v2 crosstab, bank impact and bank-swap history) are left out; each
  section ends with a short QA.

**Rebuild pass (`PASS_SUFFIX = "_v5"`):** every table is written under a new name — `pkg_cpty_dim_v5`,
`pkg_cpty_entity_map_v5`, `pkg_fi_routing_v5`, `pkg_fi_name_map_v5`, `pkg_cust_dim_v5`, `pkg_cpty_replacement_v5` —
so the tables the apps read stay as they are until the apps move; then the old ones can be dropped.

**Changes in v2 (2026-11-17, "PKG Rebuild Spec")**
- **Brands named after people merge across banks** (§3, rule `business_pair`): a business key that is not distinctive by
  its words, whose first two words form a rare pair (in at most `BRAND_PAIR_DF_MAX` distinct keys) and that has at least
  `BRAND_PAIR_MIN_IDS` ids at `BRAND_PAIR_MIN_FIS` or more banks, plus the hand list `ALWAYS_MERGE_KEYS`. §3 lists the new
  merges for review.
- **Account-type flags** (§4b, §5): `name_group` (PNC own / processors and servicers / other named large accounts, from
  `NAME_GROUPS`) on the map and on `pkg_cust_dim`; `is_pnc_own` on `pkg_cust_dim`; `is_rotator` on the map: a merged
  entity whose customers each see it through 2 or more banks in the same month (processors moving money between their
  own banks), in at least `ROT_MIN_MONTHS` of its last `ROT_LAST` active months.
- **Replacement map** (§5b, new table `pkg_cpty_replacement`): for TM customers, a steady outside counterparty that stops
  is linked to the new one that took its place (amount within `REPL_AMT_TOL`, start within `REPL_WINDOW` month of the
  month after the stop, and a name, bank or last-name link).

**How it runs.** Each table is written to a staging folder; §6 publishes all of them together at the end, so a run that
stops halfway leaves the previous tables in place. Rerun all cells after a failure.

**Downstream.** The counterparty and customer metrics notebooks treat new versions of these tables as a reason to
rebuild from 2024-01.

## Settings

In [ ]:
# ---- Settings ------------------------------------------------------------------------------------
DB         = "bdahd01p_dlcdi1_cdi_tm"
CUST_TABLE = "dsihd01p_dsi.neo4j_customer"
SHUFFLE_PARTITIONS = 2000

# ---- Counterparty dimension (pkg_cpty_dim): flags and name key, unchanged
NAME_KEY_MINLEN = 4
DIM_BIZ = (r"\b(INC|INCORPORATED|LLC|L L C|CO|CORP|CORPORATION|COMPANY|LTD|LIMITED|LP|LLP|PC|PLLC|PA|NA|BANK|TRUST|FUND|"
           r"FOUNDATION|ASSOC|ASSOCIATION|SERVICES?|GROUP|HOLDINGS|PARTNERS|UNIVERSITY|COLLEGE|SCHOOL|DISTRICT|CITY|COUNTY|"
           r"STATE|TOWNSHIP|BOROUGH|HOSPITAL|CENTER|CHURCH|DEPT|DEPARTMENT|TREASURER|INSURANCE|SUPPLY|ENTERPRISES|"
           r"INTERNATIONAL|AUTHORITY|SYSTEMS|SOLUTIONS|TECHNOLOGIES|MANAGEMENT|PROPERTIES|REALTY|CONSTRUCTION)\b")
DIM_LEGAL = r"\b(N A|INC|INCORPORATED|LLC|L L C|CO|CORP|CORPORATION|COMPANY|LTD|LIMITED|LP|LLP|PC|PLLC|PA|NA|THE)\b"
DIM_FI_LIKE = r"\b(BANK|BANCORP|BANCSHARES|N A|NATIONAL ASSOCIATION|CREDIT UNION|FCU|SAVINGS|TRUST COMPANY)\b"
DIM_PLACEHOLDER_KEYS = ["BANK ACCOUNT NAME", "ACCOUNT NAME", "UNKNOWN", "UNKNOWN PAYEE", "NONE", "NULL", "N A", "NA",
                        "TEST", "PAYEE", "VENDOR", "CUSTOMER", "ACCOUNT", "CASH", "CHECK", "DEPOSIT", "TRANSFER",
                        "PAYMENT", "REFUND", "MISC", "MISCELLANEOUS", "VARIOUS", "NAME"]

# ---- ent_l1: a 6C id joins its 5C id when at least this share of its dedup pairs point to that 5C id
XW_MIN_PURITY = 0.8

# ---- Typer v2 and entity map v2 (unchanged unless noted)
FN_MIN_IDS     = 200        # a first name must lead at least this many P2P names …
FN_LEAD_RATIO  = 3.0        # … and lead ≥ this many times more often than it trails
KEY_MIN_LEN    = 4          # shortest cleaned business key allowed to merge
WIRE_INST_MIN  = 1_000_000  # wire-only ids with at least this many lifetime dollars are institutions
NUM_SHARE      = 0.8        # digits / alphanumerics at or above this → numeric_name
PERSON_MAX_AMT = 10_000_000 # person rules apply only below this lifetime amount
CUT_NAME_LENS  = [16, 20, 22] # NEW: name lengths where the source cuts names (NACHA company 16, individual 22; some 20)
CUT_MIN_LEN    = 4          # NEW: shortest cut-off last word matched to the start of a business word
VOCAB_MIN_IDS  = 300        # learned business word: in ≥ this many legal-suffix / org-word names …
VOCAB_RATIO    = 20.0       # … and ≥ this many times more often than in P2P person names
DF_MAX         = 300        # a key is distinctive if it has a word (≥ 4 letters) in ≤ this many distinct keys
CONC_MIN       = 20         # … or if it averages ≥ this many ids per FI
KEY_IGNORE_SPACES = True    # NEW: spaces ignored in merge keys (HARLEYDAVIDSON = HARLEY DAVIDSON)
NO_MERGE_KEYS  = []
PASS_SUFFIX = "_v5"   # every table this rebuild pass writes ends in _v5, so the tables the apps read (Prospect Finder,
                      # Data Center Network Explorer, TPPP finder) stay as they are until the apps move; then drop the old ones
BRAND_PAIR_MERGE   = True   # v2: brands named after people merge across banks (rule business_pair): the key's first word is
                            # a learned first name, its second word is not a common surname or a generic word …
BRAND_PAIR_DF_MAX  = 50     # … the two words appear in at most this many distinct keys …
BRAND_PAIR_MIN_IDS = 20     # v2: … the key (all spellings) has at least this many ids …
BRAND_PAIR_MIN_FIS = 3      # v2: … at this many banks or more …
BRAND_PAIR_CONC    = 4      # … and at least this many ids per bank (one brand's ids gather at few banks; churches, motels and
                            # restaurants that share a name are spread one or two per bank: TRINITY EPISCOPAL 96 ids, 68 banks)
ALWAYS_MERGE_KEYS  = ["HARLEY DAVIDSON", "JOHN DEERE", "MORGAN STANLEY", "JOHNSON CONTROLS", "CHARLES SCHWAB",
                      "EDWARD JONES", "DUKE ENERGY"]   # v2: hand list: keys starting with these merge across banks
SHORT_KEYS_OK  = ["UPS", "BP", "DHL", "HP", "IBM", "GE", "3M", "CVS", "ATT", "UBS", "AIG", "HSN", "QVC", "KFC", "UMR"]
BRAND_BEH_SHARE = 0.8
KEY_ALIASES = {r"^(AMAZON|AMZN)\b.*": "AMAZON", r"^(MICROSOFT|MSFT)\b.*": "MICROSOFT",
               r"^(SOUTHWES\w*|SWA)\b.*": "SOUTHWEST AIRLINES", r"^(FACEBK|FACEBOOK)\b.*": "FACEBOOK"}
KEEP_NUM_AFTER = ["POST", "LOCAL", "CHAPTER", "LODGE", "TROOP", "COUNCIL", "DISTRICT", "WARD"]
TEST_NAME_RE   = r"SMOKE ?TEST|\bTEST(ER)? ?(USER|ACCOUNT|ACCT|NAME|\d+)\b"
LEGAL = ["INC", "INCORPORATED", "LLC", "LLP", "LP", "LLLP", "LTD", "LIMITED", "CORP", "CORPORATION", "CO", "COMPANY",
         "PC", "PLLC", "PA", "PLC", "SCSP", "SCS", "SARL", "SA", "AG", "GMBH", "BV", "NV", "SPA", "SRL", "PTE", "PTY",
         "ULC", "DBA", "LTDA", "SAS", "KG", "AB", "OYJ", "NA"]
ORG = ["HOLDINGS", "HOLDING", "FUND", "FUNDS", "CAPITAL", "PARTNERS", "PARTNERSHIP", "INVESTORS", "INVESTMENT",
       "INVESTMENTS", "MANAGEMENT", "AGGREGATOR", "ACCOUNT", "COLLECTION", "COLLECTIONS", "ESCROW", "TITLE",
       "SETTLEMENT", "TRUST", "TRUSTEE", "ESTATE OF", "CLINIC", "HOSPITAL", "HEALTH", "HEALTHCARE", "MEDICAL",
       "DENTAL", "PHARMACY", "UNIVERSITY", "COLLEGE", "SCHOOL", "SCHOOLS", "ACADEMY", "CHURCH", "MINISTRIES",
       "MINISTRY", "PARISH", "DIOCESE", "TEMPLE", "MOSQUE", "ASSOC", "ASSOCIATION", "ASSN", "ASSOCIATES", "FOUNDATION",
       "SOCIETY", "INSTITUTE", "CENTER", "CENTRE", "CLUB", "LEAGUE", "COUNCIL", "COMMITTEE", "UNION", "SERVICES",
       "SERVICE", "SOLUTIONS", "GROUP", "ENTERPRISES", "ENTERPRISE", "INDUSTRIES", "INTERNATIONAL", "SYSTEMS",
       "TECHNOLOGIES", "TECHNOLOGY", "CONSTRUCTION", "CONTRACTING", "CONTRACTORS", "BUILDERS", "REALTY",
       "PROPERTIES", "PROPERTY", "APARTMENTS", "HOMEOWNERS", "HOA", "CONDO", "CONDOMINIUM", "INSURANCE", "AGENCY",
       "MORTGAGE", "LENDING", "LOAN", "LOANS", "FINANCIAL", "FINANCE", "PAYROLL", "PAYMENTS", "TREASURY",
       "DEPARTMENT", "DEPT", "COUNTY", "CITY OF", "STATE OF", "TOWN OF", "VILLAGE OF", "TOWNSHIP", "BOROUGH",
       "AUTHORITY", "DISTRICT", "COMMISSION", "COMMONWEALTH", "GOVERNMENT", "UNITED STATES", "US DEPT", "IRS",
       "UTILITIES", "UTILITY", "ELECTRIC", "ENERGY", "POWER", "WATER", "GAS", "TELECOM", "WIRELESS",
       "COMMUNICATIONS", "MOTORS", "AUTOMOTIVE", "AUTO", "RESTAURANT", "CAFE", "GRILL", "PIZZA", "MARKET", "STORE",
       "STORES", "SHOP", "SUPPLY", "SUPPLIES", "DISTRIBUTION", "DISTRIBUTORS", "WHOLESALE", "LOGISTICS",
       "TRANSPORT", "TRANSPORTATION", "TRUCKING", "FREIGHT", "EXPRESS", "FARM", "FARMS", "LAW", "LEGAL", "ATTORNEYS",
       "CPA", "CPAS", "ACCOUNTING", "CONSULTING", "CONSULTANTS", "DESIGN", "STUDIO", "STUDIOS", "MEDIA", "MARKETING",
       "PRODUCTIONS", "VENTURES", "EQUITY", "ADVISORS", "ADVISORY", "SECURITIES", "BROKERAGE", "REIT", "LEASING",
       "RENTAL", "RENTALS", "PLUMBING", "ELECTRICAL", "ROOFING", "HVAC", "LANDSCAPING", "CLEANING", "CORPORATE",
       "TOPCO", "BIDCO", "HOLDCO", "OPCO", "ACQUISITION", "MERGER", "SUB", "SPV", "NOMINEE", "OPERATING", "PRODUCTS",
       "MANUFACTURING", "PROCESSING", "PROCESSOR", "PROCESSORS", "LABS", "LABORATORIES", "PHARMACEUTICALS", "SOFTWARE",
       "NETWORKS", "AIRLINES", "HOTEL", "HOTELS", "RESORT", "CASINO", "THEATRE", "THEATER", "MUSEUM", "LIBRARY",
       "PROGRAM", "PROJECT", "PLAN", "FEDERAL", "NATIONAL", "AMERICA", "AMERICAN", "UNITED", "GENERAL", "CREDIT UNION",
       "REVENUE", "TREASURER", "CUSTOMERS", "GATEWAY", "GATEWAYS", "MERCHANT", "RECEIPT", "WIRES", "TRANSACTION",
       "TRANSACTIONS", "CASH CARRY", "AIRCRAFT", "ENGINES", "AIR LINES", "INTERACTIVE", "DISTRIBUTORS"]
ORG_BRANDS = ["PAYMENTECH", "PAYPAL", "WORLDPAY", "FISERV"]
CHANNEL = ["OTP", "IVR", "WEB", "PPD", "CCD", "WEBPAY", "AUTOPAY", "EPAY", "ONLINE", "PMT", "PMTS", "PYMT", "PAYMENT",
           "PAYMENTS", "BILLPAY", "BILL PAY", "ACH", "DEBIT", "RECURRING"]
STOP = ["AND", "OF", "THE", "FOR", "DE", "LA", "DEL", "LOS", "LAS", "Y", "DA", "DOS", "VAN", "DER", "MC", "ST", "LE",
        "DI", "AL", "EL", "BIN", "BEN", "ABU"]
PERSON_EXTRA = ["MR", "MRS", "MS", "MISS", "DR", "JR", "SR", "II", "III", "IV", "MD", "DO", "DDS", "DMD", "DVM", "DC",
                "OD", "RN", "NP", "PHD", "ESQ"]
NOT_FIRST = ["THE", "AND", "OF", "MY", "YOUR", "OUR", "NEW", "FIRST", "BEST", "SUPER", "GREAT", "AMERICAN", "NATIONAL",
             "US", "USA", "ALL", "ONE", "BIG", "LITTLE", "GOOD", "TOP", "PRO", "ROYAL", "GOLDEN", "BLUE", "RED",
             "GREEN", "BLACK", "WHITE", "NORTH", "SOUTH", "EAST", "WEST", "CENTRAL", "SAINT", "ST", "MOUNT", "HOLY",
             "GRACE", "LOVE", "PAYPAL", "VENMO", "ZELLE", "CASH"]
GENERIC_KEYS = ["PAYMENT", "PAYMENTS", "TRANSFER", "DEPOSIT", "REFUND", "CHECK", "WIRE", "ACH", "ONLINE", "MOBILE",
                "BILL", "BILLPAY", "PAYEE", "VENDOR", "CUSTOMER", "CLIENT", "OWNER", "TENANT", "LANDLORD", "USER",
                "CHECKING", "SAVINGS", "ACCT", "ACCOUNT", "MONEY MARKET"]
PLACEHOLDERS = ["BANK ACCOUNT NAME", "UNKNOWN", "UNKNOWN NAME", "NONE", "NA", "N A", "NULL", "TEST", "CUSTOMER",
                "BENEFICIARY", "PAYEE", "ACCOUNT HOLDER", "NAME", "NO NAME", "ON FILE", "CASH", "VARIOUS", "XXX",
                "USER", "CHECKING", "SAVINGS", "ACCT", "CHECKING ACCOUNT", "SAVINGS ACCOUNT", "MY ACCOUNT"]
STREET = ["ROAD", "RD", "STREET", "ST", "AVENUE", "AVE", "AV", "BOULEVARD", "BLVD", "DRIVE", "DR", "LANE", "LN",
          "HIGHWAY", "HWY", "WAY", "COURT", "CT", "PLACE", "PL", "PARKWAY", "PKWY", "CIRCLE", "CIR", "TERRACE", "TER",
          "TRAIL", "TRL", "PIKE", "ROUTE", "RT", "RTE", "SQUARE", "SQ", "SUITE", "STE", "APT", "UNIT", "FLOOR", "FL"]

# ---- Canonical bank names (unchanged unless noted)
FI_FIXES = {"ALLLY BANK": "ALLY BANK"}          # raw FI name → corrected name (exact match)
# NEW: (pattern for raw names to fix, pattern for the target name). Every raw FI name matching the first pattern takes
# the raw FI name matching the second pattern that has the most dollars. Banks and credit unions never match. §4 shows
# what was matched.
VA_NAME_RE       = r"^(?!.*\b(CREDIT UNION|FCU|BANK)\b).*\bVETERANS AFFAIRS\b"
TREASURY_NAME_RE = r"^(?!.*\b(CREDIT UNION|FCU|BANK)\b).*\b(DEPARTMENT OF THE TREASURY|DEPT OF THE TREASURY|US TREASURY)\b"
FI_FIX_RULES = [(VA_NAME_RE, TREASURY_NAME_RE)]
FI_PARTNER_RE = (r"^.+?\s+(?:WITH|PARTNERING WITH|POWERED BY|IN PARTNERSHIP WITH|THROUGH)\s+"
                 r"(.*\b(?:BANK|CREDIT UNION|TRUST|N A|NA|FSB|SSB|SAVINGS|BANCORP|FCU)\b.*)$")
FI_DIVISION_RE = r"^.+?\s+(?:ADO|A DIVISION OF|DIVISION OF|A DIV OF|DIV OF)\s+(.{4,})$"
FI_KEEP_RAW_ROUTINGS = []
NAME_MIN_SHARE = 0.9
FI_STRIP = ["THE", "NA", "NATIONAL ASSOCIATION", "INC", "CORP", "CORPORATION", "CO", "COMPANY", "FSB", "SSB",
            "FEDERAL CREDIT UNION", "CREDIT UNION", "FCU", "LLC"]

# ---- v2: account-type flags. Name groups are matched on the upper-case name (customers: customer_name; counterparties:
# the cleaned name), first match wins. Edit the lists freely; a change rebuilds the metrics downstream.
NAME_GROUPS = {
    # names that start with PNC (trust accounts with PNC as trustee, "... FBO PNC ...", "SMBC NY PNC" are not PNC's own)
    "PNC own": r"^(THE )?PNC\b|^PNCINT\b|^DUE (FROM|TO) PNC\b",
    "processors and servicers": (r"\b(PAYLOCITY|PAYCOM|TELECHECK|FISERV|STRIPE|PAYPAL|PAYCHEX|CERIDIAN|DAYFORCE|SITUS|"
                                 r"WORLDPAY|GLOBAL PAYMENTS|FIRST DATA|BILL ?COM|INTUIT|EMPOWER|AFFIRM|EARNIN|"
                                 r"GUSTO|PAYCOR|TRINET|INSPERITY|JUSTWORKS)\b|^(SQUARE|BLOCK) INC\b|"
                                 r"\bADP (INC|LLC|CLIENT|PAYROLL|TOTALSOURCE|TAX|WAGE|FSA|SCREENING|RETIREMENT|BENEFIT)"),
    "other named large accounts": r"^FMR (LLC|CORP|CORPORATION)\b|\bWALKER (AND )?DUNLOP\b",
}
ROT_TOP_SHARE  = 0.9    # a customer pays / is paid by the entity through 2+ banks in a month, top bank under this share …
ROT_PAIR_SHARE = 0.5    # … an entity month is split when such customers carry at least this share of its dollars …
ROT_LAST       = 6      # … looking at the entity's last this many active months …
ROT_MIN_MONTHS = 3      # … and the entity is a rotator with at least this many split months (merged N: entities only) …
ROT_MAX_BANKS  = 10     # … and no customer sees it through more banks than this in a month (more = a merged group of separate
                        # payers such as franchisees: TACO BELL 737065 at 150 banks, not one payer moving its money)

# ---- v2: stopped-payer replacement map (TM customers, outside counterparties)
REPL_STEADY_MIN = 4     # old counterparty paid in at least this many of the 6 months before the stop
REPL_SILENT     = 2     # stop = no payment in this many months after the last payment (or the data ends)
REPL_NEW_GAP    = 12    # new counterparty: not seen for the customer in this many months before its start
REPL_NEW_HIST   = 6     # in the first months of history a counterparty counts as new only from this month index on
REPL_WINDOW     = 1     # new counterparty starts within this many months of the stop month
REPL_AMT_TOL    = 0.05  # new monthly amount (mean of its first 2 months) within this share of the old median of 3
REPL_BUSY_MIN   = 3     # customer-month with at least this many new counterparties and above the customer's p90 = busy:
                        # there a last-name link alone is not enough (a business-name or bank link is needed)
REPL_MAX_PER_MONTH = 2000   # customer-months with more steady stops or more new counterparties than this are left out
                            # (collectors and processors: too many same-amount payers to tell a replacement)

# ---- Customer dimension (unchanged)
NAICS_PLACEHOLDERS = ["******", "-1", "0", "00", "000000", "999999", "UNKNOWN", "N/A", "NA", "NULL", "NONE"]

In [ ]:
# ---- Setup ---------------------------------------------------------------------------------------
# Compatibility shims: Cloudera SPARK3 PySpark is older than the NumPy / pandas in the conda env.
import numpy as np
import pandas as pd
import warnings
for _alias, _target in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
if "bool" not in np.__dict__:
    np.bool = np.bool_
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
for _cat in (ResourceWarning, DeprecationWarning, FutureWarning):
    warnings.filterwarnings("ignore", category=_cat)

import re, time, datetime as dt, html as _html
from IPython.display import display, HTML
from pyspark.sql import SparkSession, Window, functions as F

HDFS_REPLICATION = 2          # copies of every file this notebook writes (cluster default 3); no effect on the numbers
spark = (SparkSession.builder.appName("pkg_reference_data")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "4g").config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g").config("spark.executor.memoryOverhead", "2g")
         .config("spark.dynamicAllocation.enabled", "true").config("spark.dynamicAllocation.maxExecutors", "512")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.sql.adaptive.enabled", "true").config("spark.sql.adaptive.skewJoin.enabled", "true")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.hadoop.dfs.replication", str(HDFS_REPLICATION))
         .enableHiveSupport().getOrCreate())
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
spark.sparkContext.setLogLevel("ERROR")
spark._jsc.hadoopConfiguration().set("dfs.replication", str(HDFS_REPLICATION))   # also when the session existed already
# The cluster's Atlas lineage hook can't reach its Kafka topic and prints the full query plan on every write.
# It doesn't affect the data; its loggers are switched off so the cell outputs stay readable.
_QUIET = ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage", "org.apache.kafka",
          "org.apache.spark.sql.catalyst.util.package")
try:
    _L = spark._jvm.org.apache.logging.log4j
    for _n in _QUIET:
        _L.core.config.Configurator.setLevel(_n, _L.Level.OFF)
except Exception:
    try:
        _L1 = spark._jvm.org.apache.log4j
        for _n in _QUIET:
            _L1.LogManager.getLogger(_n).setLevel(_L1.Level.OFF)
    except Exception as _e:
        print("could not silence the Atlas logs:", str(_e)[:120])
pd.set_option("display.max_rows", 300); pd.set_option("display.max_columns", 80)

# ---- one consolidated HTML output per cell
def tbl(pdf, pct=(), dec=()):
    pdf = pdf.copy()
    fmt = {}
    for c in pdf.columns:
        if c in pct:                                   fmt[c] = "{:.1%}"
        elif c in dec:                                 fmt[c] = "{:,.2f}"
        elif pd.api.types.is_bool_dtype(pdf[c]):       continue
        elif pd.api.types.is_integer_dtype(pdf[c]):    fmt[c] = "{:,}"
        elif pd.api.types.is_float_dtype(pdf[c]):      fmt[c] = "{:,.0f}"
    sty = pdf.style.format(fmt, na_rep="—")
    try:
        sty = sty.hide(axis="index")
    except AttributeError:
        sty = sty.hide_index()
    sty = sty.set_table_styles([{"selector": "th, td", "props": "padding: 2px 8px; font-size: 12px;"}])
    return sty.to_html()
def sec(title, body):
    return f"<h4 style='margin:14px 0 4px 0'>{_html.escape(title)}</h4>{body}"
def para(text):
    return f"<p style='margin:4px 0; font-size:13px'>{_html.escape(text)}</p>"
# ---- progress while a cell runs: one panel per cell (step, minutes, Spark tasks), refreshed every 30 s and replaced
# by the cell's result when it finishes
import threading
_PROG = {"h": None, "t0": None, "title": "", "step": ""}
def _stages():
    try:
        st = spark.sparkContext.statusTracker()
        info = [x for x in (st.getStageInfo(i) for i in st.getActiveStageIds()) if x is not None]
        if not info:
            return "between stages"
        return " · ".join(f"stage {x.stageId}: {x.numCompletedTasks:,} of {x.numTasks:,} tasks"
                          for x in sorted(info, key=lambda x: x.stageId)[:3])
    except Exception:
        return "—"
def _prog_html():
    return HTML(f"<p style='margin:4px 0; font-size:13px'><b>{_html.escape(_PROG['title'])}</b> · "
                f"{_html.escape(_PROG['step'])} · {(time.time() - _PROG['t0']) / 60:.1f} min<br>Spark: {_stages()}"
                f"<br><span style='font-size:11px'>updated {dt.datetime.now().strftime('%H:%M:%S')}</span></p>")
def progress(title, step):
    """Show (or update) this cell's progress panel."""
    if _PROG["h"] is None:
        _PROG.update(t0=time.time(), h=display(HTML(""), display_id=True))
    _PROG.update(title=title, step=step)
    _PROG["h"].update(_prog_html())
def _heartbeat():
    while True:
        time.sleep(30)
        try:
            if _PROG["h"] is not None:
                _PROG["h"].update(_prog_html())
        except Exception:
            pass
if not globals().get("_HB_STARTED"):
    threading.Thread(target=_heartbeat, daemon=True).start()
    _HB_STARTED = True
def out(*parts):
    """The cell's one result: replaces its progress panel when there is one."""
    h, _PROG["h"] = _PROG["h"], None
    res = HTML("<div>" + "".join(p for p in parts if p) + "</div>")
    if h is not None:
        h.update(res)
    else:
        display(res)

# ---- HDFS (Hadoop FileSystem API through the JVM)
HCONF = spark._jsc.hadoopConfiguration()
JPath = spark._jvm.org.apache.hadoop.fs.Path
fs_for = lambda p: JPath(p).getFileSystem(HCONF)
exists = lambda p: fs_for(p).exists(JPath(p))
def mkdirs(p):
    fs_for(p).mkdirs(JPath(p))
def delete(p):
    return fs_for(p).delete(JPath(p), True) if exists(p) else False
def rename(a, b):
    assert not exists(b), f"rename target already exists: {b}"
    if not fs_for(a).rename(JPath(a), JPath(b)):
        raise RuntimeError(f"rename failed: {a} -> {b}")
def size_gb(p):
    return fs_for(p).getContentSummary(JPath(p)).getLength() / 1e9 if exists(p) else 0.0

TIMES = {}
class timed:
    def __init__(self, name): self.name = name
    def __enter__(self): self.t0 = time.time()
    def __exit__(self, *a): TIMES[self.name] = (time.time() - self.t0) / 60

def nz(c):
    """String, trimmed, blank -> NULL."""
    s = F.trim(F.col(c).cast("string"))
    return F.when(s == "", F.lit(None)).otherwise(s)

print(f"Spark {spark.version} | pandas {pd.__version__} | numpy {np.__version__}")

## §0 Inputs

The edge tables must exist, and every month of `pkg_cpty_month` must have been published by `pkg_edges_refresh.ipynb`
(build marker present). The customer table must have the columns the customer dimension uses.

In [ ]:
# ---- §0 Inputs -------------------------------------------------------------------------------------
T_EDGE_M, T_CPTY_MONTH, T_DEDUP = f"{DB}.pkg_edge_monthly", f"{DB}.pkg_cpty_month", f"{DB}.pkg_dedup_5c6c"
OUT_KEYS = ["cpty_dim", "cpty_entity_map_v2", "fi_routing", "fi_name_map", "cust_dim", "cpty_replacement"]
TNAME = {k: ("pkg_cpty_entity_map" if k == "cpty_entity_map_v2" else f"pkg_{k}") + PASS_SUFFIX for k in OUT_KEYS}
T = {k: f"{DB}.{TNAME[k]}" for k in OUT_KEYS}
DB_LOC = [r[1] for r in spark.sql(f"DESCRIBE DATABASE {DB}").collect() if r[0].lower() == "location"][0].rstrip("/")
P = {k: f"{DB_LOC}/{TNAME[k]}" for k in OUT_KEYS}
STAGE = f"{DB_LOC}/_staging_pkg_reference{PASS_SUFFIX}"
TMP   = f"{DB_LOC}/_tmp_pkg_reference{PASS_SUFFIX}"
S = {k: f"{STAGE}/{k}" for k in OUT_KEYS}

def table_exists(t):
    db, name = t.split(".")
    return len(spark.sql(f"SHOW TABLES IN {db} LIKE '{name}'").collect()) > 0
def table_location(t):
    return [r[1] for r in spark.sql(f"DESCRIBE TABLE EXTENDED {t}").collect() if r[0] == "Location"][0].rstrip("/")

PROBLEMS = []
NEED = {T_EDGE_M: ["src_id", "src_kind", "dst_id", "dst_kind", "is_self", "amount", "month"],
        T_CPTY_MONTH: ["cpty_id", "cpty_id_src", "category", "cpty_type", "cpty_name", "fi", "n_txn", "amount", "month"],
        T_DEDUP: ["cpty_id_drop", "cpty_id_keep", "month"],
        CUST_TABLE: ["mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "party_type", "customer_start_dt",
                     "naics_cd", "naics_desc", "load_dt"]}
for t, cols in NEED.items():
    if not table_exists(t):
        PROBLEMS.append({"problem": "table missing", "detail": t}); continue
    have = {c.lower() for c in spark.table(t).columns}
    for c in cols:
        if c not in have:
            PROBLEMS.append({"problem": "column missing", "detail": f"{t}.{c}"})

MONTHS = []
if table_exists(T_CPTY_MONTH):
    MONTHS = sorted(r[0].split("=", 1)[1] for r in spark.sql(f"SHOW PARTITIONS {T_CPTY_MONTH}").collect())
    _cm_loc = table_location(T_CPTY_MONTH)
    _unpub = [m for m in MONTHS if not exists(f"{_cm_loc}/month={m}/_PKG_BUILD")]
    if _unpub:
        PROBLEMS.append({"problem": "months not published by pkg_edges_refresh",
                         "detail": ", ".join(_unpub)})
if PROBLEMS:
    out(sec(f"§0 stopped: {len(PROBLEMS)} problem(s)", tbl(pd.DataFrame(PROBLEMS))))
    raise AssertionError(f"§0: {len(PROBLEMS)} problem(s), see the table above")

delete(STAGE); delete(TMP)                       # leftovers of an earlier run
_ex = pd.DataFrame([{"table": T[k], "exists": table_exists(T[k]),
                     "location": table_location(T[k]) if table_exists(T[k]) else P[k]} for k in OUT_KEYS])
out(sec(f"Inputs: edge tables {MONTHS[0]} → {MONTHS[-1]} ({len(MONTHS)} months, all published)",
        tbl(pd.DataFrame([{"input": t} for t in NEED]))),
    sec("Tables this notebook replaces in §6", tbl(_ex)))
_S0_OK = True

## §1 Counterparty dimension: `pkg_cpty_dim`

One row per counterparty id over every month of `pkg_cpty_month`: dominant name, bank and type (by transactions), activity
span, categories, payee / payer flags, and the v1 name flags (`is_business`, `name_is_fi`, `is_placeholder`) that the
typer in §2 reads. Same logic as the old edge notebook.

In [ ]:
# ---- §1 Counterparty dimension ---------------------------------------------------------------------
assert globals().get("_S0_OK"), "§0 did not finish: fix the problem it shows, then run all cells again"

clean = lambda c: F.trim(F.regexp_replace(F.regexp_replace(F.upper(c), "[^A-Z0-9 ]", " "), r"\s+", " "))
def raw_key(c):
    return F.trim(F.regexp_replace(F.regexp_replace(clean(c), DIM_LEGAL, " "), r"\s+", " "))

with timed("§1 counterparty dimension"):
    progress("§1 Counterparty dimension", f"aggregating pkg_cpty_month and writing {TNAME['cpty_dim']}")
    cm_all = spark.table(T_CPTY_MONTH)
    def dominant(col):
        w = Window.partitionBy("cpty_id").orderBy(F.desc("n"), F.col(col))
        return (cm_all.filter(F.col(col).isNotNull()).groupBy("cpty_id", col).agg(F.sum("n_txn").alias("n"))
                      .withColumn("r", F.row_number().over(w)).filter("r = 1").select("cpty_id", col))
    nvar = (cm_all.filter(F.col("cpty_name").isNotNull()).groupBy("cpty_id")
                  .agg(F.countDistinct("cpty_name").alias("n_name_variants")))
    letter = F.regexp_extract(F.split("category", r"\.").getItem(0), "([ABC])$", 1)
    base = cm_all.groupBy("cpty_id").agg(
               F.max("cpty_id_src").alias("cpty_id_src"), F.sum("n_txn").alias("n_txn"),
               F.sum("amount").cast("decimal(22,2)").alias("amount"),
               F.min("month").alias("first_month"), F.max("month").alias("last_month"),
               F.countDistinct("month").alias("active_months"),
               F.concat_ws(",", F.sort_array(F.collect_set(F.split("category", r"\.").getItem(0)))).alias("categories"),
               F.max((letter == "B").cast("int")).cast("boolean").alias("is_payee"),
               F.max((letter == "C").cast("int")).cast("boolean").alias("is_payer"))
    rk = raw_key(F.col("cpty_name"))
    dim = (base.join(dominant("cpty_name"), "cpty_id", "left").join(dominant("fi"), "cpty_id", "left")
               .join(dominant("cpty_type"), "cpty_id", "left").join(nvar, "cpty_id", "left")
               .withColumn("is_business",    F.coalesce(clean(F.col("cpty_name")).rlike(DIM_BIZ), F.lit(False)))
               .withColumn("name_is_fi",     F.coalesce(clean(F.col("cpty_name")).rlike(DIM_FI_LIKE), F.lit(False)))
               .withColumn("is_placeholder", F.coalesce(rk.isin(DIM_PLACEHOLDER_KEYS)
                                                        | clean(F.col("cpty_name")).isin(DIM_PLACEHOLDER_KEYS), F.lit(False)))
               .withColumn("name_key", F.when(~F.col("is_placeholder") & (F.length(rk) >= NAME_KEY_MINLEN), rk)))
    dim.write.mode("overwrite").parquet(S["cpty_dim"])
    DIM = spark.read.parquet(S["cpty_dim"])

progress("§1 Counterparty dimension", "checks")
q = DIM.agg(F.count("*").alias("ids"), F.countDistinct("cpty_id").alias("distinct_ids"),
            F.sum("amount").cast("double").alias("amount"),
            F.sum(F.col("is_business").cast("long")).alias("business_like"),
            F.sum(F.col("name_is_fi").cast("long")).alias("bank_like_names"),
            F.sum(F.col("is_placeholder").cast("long")).alias("placeholder_names"),
            F.min("first_month").alias("first_month"), F.max("last_month").alias("last_month")).toPandas()
assert int(q.ids[0]) == int(q.distinct_ids[0]), "pkg_cpty_dim must have one row per cpty_id"
out(sec(f"§1 {TNAME['cpty_dim']} (staged, {TIMES['§1 counterparty dimension']:.1f} min)", tbl(q)))

## §2 `ent_l1` and name typer v2

- **`ent_l1`:** a 6C P2P id is folded into the 5C id it was paired with in the 5C/6C dedup, when at least
  `XW_MIN_PURITY` of its pairs point to that id. Every other id is its own `ent_l1`.
- **Typer v2:** every id gets a `name_class` (business / person / bank / numeric_name / placeholder / no_name /
  unknown) and the rule that set it. First names and learned business words come from the data. Same rules as
  `pkg_cpty_entity_map_v2.ipynb`, plus three (Nov 2026):
  - a leading number glued to the name (`0007JENNIFER HWANG`) is removed before typing;
  - `ampersand`: a name with `&`, no comma, and no first name in front (`BROWN & CALDWELL`) is a business;
  - `cut_word`: a name cut by the source (length in `CUT_NAME_LENS`) whose last word is the start of a business word
    (`SYNERGETIC COMMUNICATI`, `EMBLEM GRAYSON APARTME`) is a business, unless it starts with a first name or has a
    comma. Cut-off words that are common first names or surnames in P2P names are not used.

In [ ]:
# ---- §2 ent_l1 and typer v2 ------------------------------------------------------------------------
assert globals().get("_S0_OK"), "§0 did not finish: fix the problem it shows, then run all cells again"

def wre(words):                       # whole-word alternation
    return r"\b(" + "|".join(sorted(set(words), key=len, reverse=True)) + r")\b"

with timed("§2 ent_l1 + typer"):
    progress("§2 ent_l1 and typer v2", "1/5 ent_l1 crosswalk")
    # ent_l1: 6C → 5C crosswalk from the dedup pairs
    xw = (spark.table(T_DEDUP).filter(F.col("cpty_id_drop").isNotNull() & F.col("cpty_id_keep").isNotNull())
               .groupBy("cpty_id_drop", "cpty_id_keep").agg(F.count("*").alias("pairs")))
    wx = Window.partitionBy("cpty_id_drop").orderBy(F.desc("pairs"), "cpty_id_keep")
    xw = (xw.withColumn("tot", F.sum("pairs").over(Window.partitionBy("cpty_id_drop")))
            .withColumn("r", F.row_number().over(wx)).filter("r = 1")
            .filter(F.col("pairs") / F.col("tot") >= XW_MIN_PURITY)
            .select(F.col("cpty_id_drop").alias("cpty_id"), F.col("cpty_id_keep").alias("xw_to")))
    l1 = (DIM.select("cpty_id", "cpty_id_src").join(xw, "cpty_id", "left")
             .withColumn("ent_l1", F.coalesce("xw_to", "cpty_id")).select("cpty_id", "cpty_id_src", "ent_l1"))

    base = (DIM.select("cpty_id", F.col("cpty_name").alias("name"), "fi", "cpty_type", F.col("categories").alias("cats"),
                       F.col("amount").cast("double").alias("amt"), F.col("n_txn").cast("long").alias("n_txn"),
                       F.col("name_is_fi").cast("boolean").alias("v1_fi"),
                       F.col("is_placeholder").cast("boolean").alias("v1_ph"))
               .join(l1, "cpty_id", "inner"))

    # normalised names
    up   = F.upper(F.trim(F.col("name")))
    nm0  = F.trim(F.regexp_replace(F.regexp_replace(up, r"[^A-Z0-9]+", " "), r"\s+", " "))
    nm0  = F.regexp_replace(nm0, r"^0\d{2,3}(?=[A-Z])", "")      # NEW: 0007JENNIFER HWANG → JENNIFER HWANG
    ADDR = (r"^(.*?[A-Z].*?)\s+(?:\d{1,6}[A-Z]?\s+(?:[A-Z0-9]+\s+){0,5}?(?:" + "|".join(STREET)
            + r")\b|P\s?O\s+BOX\b|BOX\s+\d).*$")
    nm   = F.trim(F.regexp_replace(nm0, ADDR, "$1"))
    nm_c = F.regexp_replace(nm, r"\b([A-Z]) (?=[A-Z]\b)", "$1")
    nm_p = F.trim(F.regexp_replace(F.regexp_replace(nm, wre(PERSON_EXTRA), " "), r"\s+", " "))
    after_comma = F.trim(F.regexp_replace(F.regexp_replace(F.upper(F.substring_index(F.col("name"), ",", -1)),
                                                           r"[^A-Z]+", " "), r"\s+", " "))
    named = (base
             .withColumn("nm", nm).withColumn("nm_c", nm_c).withColumn("nm_p", nm_p)
             .withColumn("has_comma", F.col("name").contains(","))
             .withColumn("ft1", F.element_at(F.split(F.col("nm_p"), " "), 1))
             .withColumn("ftc", F.when(F.col("has_comma"), F.element_at(F.split(after_comma, " "), 1)))
             .withColumn("n_tok", F.size(F.split(F.col("nm_p"), " ")))
             .withColumn("alpha", F.col("nm_p").rlike(r"^[A-Z]+( [A-Z]+)*$"))
             .withColumn("biz_tok", F.col("nm_c").rlike(wre(LEGAL + ORG))))

    progress("§2 ent_l1 and typer v2", "2/5 first names from P2P names")
    # first names from P2P names: leading vs trailing counts per token
    p2p = named.where((F.col("cpty_type") == "P2P_CPTY") & F.col("alpha") & F.col("n_tok").between(2, 3) & ~F.col("biz_tok"))
    toks = F.split(F.col("nm_p"), " ")
    lead = F.when(F.col("has_comma") & F.col("ftc").isNotNull(), F.col("ftc")).otherwise(F.element_at(toks, 1))
    trail = F.when(F.col("has_comma"), F.element_at(toks, 1)).otherwise(F.element_at(toks, -1))
    fn_stats = (p2p.select(lead.alias("tok"), F.lit(1).alias("lead"), F.lit(0).alias("trail"))
                .unionByName(p2p.select(trail.alias("tok"), F.lit(0).alias("lead"), F.lit(1).alias("trail")))
                .groupBy("tok").agg(F.sum("lead").alias("lead"), F.sum("trail").alias("trail"))
                .where((F.length("tok") >= 2) & ((F.col("lead") >= FN_MIN_IDS) | (F.col("trail") >= FN_MIN_IDS)))
                .toPandas())
    PERSON_TOKS = set(fn_stats.tok)                    # NEW: frequent first or last words of P2P names (cut_word guard)
    SURNAMES = set(fn_stats[(fn_stats.trail >= FN_MIN_IDS) & (fn_stats.trail >= FN_LEAD_RATIO * fn_stats.lead.clip(lower=1))].tok)
    fn_stats = fn_stats[(fn_stats.lead >= FN_MIN_IDS) & (fn_stats.lead >= FN_LEAD_RATIO * fn_stats.trail.clip(lower=1))
                        & ~fn_stats.tok.isin(NOT_FIRST + LEGAL + ORG)]
    FIRST_NAMES = set(fn_stats.tok)
    FN_DF = F.broadcast(spark.createDataFrame(pd.DataFrame({"tok": sorted(FIRST_NAMES)}), "tok string"))

    progress("§2 ent_l1 and typer v2", "3/5 learned business words")
    # learned business words: frequent in organisation names, rare in person names
    def tok_counts(df, col, label):
        return (df.select("cpty_id", F.explode(F.array_distinct(F.split(F.col(col), " "))).alias("tok"))
                  .where(F.col("tok").rlike(r"^[A-Z]{3,}$")).groupBy("tok").agg(F.count("*").alias(label)))
    biz_src = named.where(F.col("biz_tok") & ~F.coalesce(F.col("v1_ph"), F.lit(False))
                          & ~F.coalesce(F.col("v1_fi"), F.lit(False)) & ~F.col("nm").isin(PLACEHOLDERS))
    vocab = (tok_counts(biz_src, "nm_c", "biz_ids").join(tok_counts(p2p, "nm_p", "person_ids"), "tok", "left")
             .fillna(0, ["person_ids"])
             .where((F.col("biz_ids") >= VOCAB_MIN_IDS) & (F.col("biz_ids") >= VOCAB_RATIO * (F.col("person_ids") + 1)))
             .where(~F.col("tok").isin(STOP + LEGAL + PERSON_EXTRA))
             .toPandas())
    vocab = vocab[~vocab.tok.isin(FIRST_NAMES)]
    BIZ_VOCAB = set(vocab.tok)

    TOKV = F.broadcast(spark.createDataFrame(pd.DataFrame(
        [(t, t in FIRST_NAMES, t in BIZ_VOCAB) for t in sorted(FIRST_NAMES | BIZ_VOCAB)], columns=["tok", "is_fn", "is_bv"]),
        "tok string, is_fn boolean, is_bv boolean"))
    tok_flags = (named.select("cpty_id", F.posexplode(F.slice(F.split(F.col("nm_p"), " "), 1, 8)).alias("pos", "tok"))
                      .join(TOKV, "tok", "inner")
                      .groupBy("cpty_id")
                      .agg(F.max((F.col("is_fn") & (F.col("pos") == 0)).cast("int")).alias("fn_t1"),
                           F.max((F.col("is_fn") & (F.col("pos") == 1)).cast("int")).alias("fn_t2"),
                           F.max(F.col("is_fn").cast("int")).alias("fn_any"),
                           F.max(F.col("is_bv").cast("int")).alias("bv_any")))
    progress("§2 ent_l1 and typer v2", "4/5 token flags")
    tok_flags.write.mode("overwrite").parquet(f"{TMP}/tok_flags")          # materialised: typing is not one huge stage

    # NEW cut_word: starts (≥ CUT_MIN_LEN letters) of legal, org and learned business words → the word
    _cut = {}
    for w in sorted({x for x in LEGAL + ORG + sorted(BIZ_VOCAB) if " " not in x and len(x) > CUT_MIN_LEN}):
        for k in range(CUT_MIN_LEN, len(w)):
            _cut.setdefault(w[:k], w)
    _cut = {k: v for k, v in _cut.items() if k not in PERSON_TOKS and k not in FIRST_NAMES}
    CUT_DF = F.broadcast(spark.createDataFrame(pd.DataFrame(sorted(_cut.items()), columns=["last_tok", "cut_of"]),
                                               "last_tok string, cut_of string"))

    # typer rules, in order
    fnc = FN_DF.select(F.col("tok").alias("ftc"), F.lit(True).alias("fnc"))
    alnum = F.length(F.regexp_replace(F.col("nm"), r"[^A-Z0-9]", ""))
    digits = F.length(F.regexp_replace(F.col("nm"), r"[^0-9]", ""))
    wire_only = F.col("cats").rlike(r"^4[A-C](,4[A-C])*$")
    t = (named.join(fnc, "ftc", "left").join(spark.read.parquet(f"{TMP}/tok_flags"), "cpty_id", "left")
         .withColumn("last_tok", F.when(F.length(F.trim(F.col("name"))).isin(CUT_NAME_LENS),
                                        F.element_at(F.split(F.col("nm"), " "), -1)))
         .join(CUT_DF, "last_tok", "left")
         .withColumn("fnc", F.coalesce("fnc", F.lit(False)))
         .fillna(0, ["fn_t1", "fn_t2", "fn_any", "bv_any"]))
    nc = F.regexp_replace(F.col("nm"), r"[^A-Z0-9]", "")
    fc = F.regexp_replace(F.upper(F.coalesce(F.col("fi"), F.lit(""))), r"[^A-Z0-9]", "")
    name_is_fi = ((F.length(nc) >= 6) & (F.length(fc) >= 6)
                  & ((F.expr("left(regexp_replace(upper(coalesce(fi, '')), '[^A-Z0-9]', ''), "
                             "length(regexp_replace(nm, '[^A-Z0-9]', '')))") == nc)
                     | (F.expr("left(regexp_replace(nm, '[^A-Z0-9]', ''), "
                               "length(regexp_replace(upper(coalesce(fi, '')), '[^A-Z0-9]', '')))") == fc)))
    shape = F.col("alpha") & F.col("n_tok").between(2, 4) & ~F.col("nm_p").rlike(wre(["THE", "AT", "OF", "ST"]))
    can_person = F.col("amt") < PERSON_MAX_AMT
    c = (F.when(F.col("name").isNull() | (F.col("nm") == ""), F.array(F.lit("no_name"), F.lit("no_name")))
          .when(F.coalesce(F.col("v1_ph"), F.lit(False)) | F.col("nm").isin(PLACEHOLDERS) | F.col("nm").rlike(TEST_NAME_RE),
                F.array(F.lit("placeholder"), F.lit("placeholder")))
          .when((alnum > 0) & (digits >= NUM_SHARE * alnum), F.array(F.lit("numeric_name"), F.lit("numeric")))
          .when(F.coalesce(F.col("v1_fi"), F.lit(False)), F.array(F.lit("bank"), F.lit("bank_name")))
          .when(name_is_fi, F.array(F.lit("bank"), F.lit("name_is_fi")))
          .when(F.col("nm_c").rlike(wre(LEGAL)), F.array(F.lit("business"), F.lit("legal_suffix")))
          .when(F.col("nm_c").rlike(wre(ORG + ORG_BRANDS)), F.array(F.lit("business"), F.lit("org_word")))
          .when(F.col("bv_any") == 1, F.array(F.lit("business"), F.lit("learned_word")))
          .when(F.col("cpty_type") == "NON_PNC_ACH_ORIGINATOR", F.array(F.lit("business"), F.lit("ach_originator")))
          .when(F.col("cpty_type") == "MERCHANT", F.array(F.lit("business"), F.lit("merchant")))
          .when(F.col("name").contains("&") & ~F.col("has_comma") & (F.col("fn_t1") == 0),
                F.array(F.lit("business"), F.lit("ampersand")))
          .when(F.col("cut_of").isNotNull() & ~F.col("has_comma") & (F.col("fn_t1") == 0),
                F.array(F.lit("business"), F.lit("cut_word")))
          .when(can_person & F.col("has_comma") & shape & F.col("fnc"), F.array(F.lit("person"), F.lit("comma_name")))
          .when(can_person & shape & (F.col("fn_t1") == 1), F.array(F.lit("person"), F.lit("first_name")))
          .when(can_person & shape & (F.col("fn_t2") == 1), F.array(F.lit("person"), F.lit("last_first")))
          .when(can_person & F.col("alpha") & F.col("nm_p").contains(" AND ") & (F.col("n_tok") <= 8)
                & (F.col("fn_any") == 1), F.array(F.lit("person"), F.lit("joint_names")))
          .when(wire_only & (F.col("amt") >= WIRE_INST_MIN), F.array(F.lit("business"), F.lit("wire_institution")))
          .when(~can_person, F.array(F.lit("business"), F.lit("high_value")))
          .when((F.col("cpty_type") == "P2P_CPTY") & shape, F.array(F.lit("person"), F.lit("p2p_shape")))
          .otherwise(F.array(F.lit("unknown"), F.lit("unknown"))))
    # merge key: legal and channel words and reference numbers removed; digit-only names keep their digits
    k = F.regexp_replace(F.col("nm_c"), r"\bAT T\b", "ATT")
    k = F.regexp_replace(k, r"\b(" + "|".join(KEEP_NUM_AFTER) + r") (\d+)\b", "$1_$2")
    k = F.regexp_replace(k, r"\b([A-Z]{3,})(\d+)\b", "$1 $2")
    k = F.regexp_replace(k, r"([A-Z])(\d{4,})", "$1 $2")
    k = F.regexp_replace(k, r"(\d{3,})([A-Z])", "$1 $2")
    k = F.regexp_replace(k, r"\b(?=(?:[A-Z]*\d){4})[A-Z0-9]+\b", " ")
    k = F.regexp_replace(k, r"\b(?=[A-Z0-9]{5,}\b)(?=[A-Z]*\d[A-Z]*\d)[A-Z0-9]+\b", " ")
    k = F.regexp_replace(k, r"\b\d+\b", " ")
    k = F.regexp_replace(k, wre(LEGAL + CHANNEL + ["THE"]), " ")
    k = F.trim(F.regexp_replace(k, r"\s+", " "))
    key = F.regexp_replace(k, r"(\s+[A-Z0-9])+$", "")
    for _pat, _canon in KEY_ALIASES.items():
        key = F.when(key.rlike(_pat), F.lit(_canon)).otherwise(key)
    progress("§2 ent_l1 and typer v2", "5/5 typing every id and writing")
    (t.withColumn("_c", c)
      .select("cpty_id", "cpty_id_src", "ent_l1", "name", "fi", "cpty_type", "cats", "amt", "n_txn",
              F.col("nm").alias("name_clean"), F.col("_c")[0].alias("name_class"), F.col("_c")[1].alias("type_reason"),
              F.when(F.col("_c")[0] == "business", key).alias("biz_key"),
              F.when(F.col("_c")[0] == "numeric_name", F.regexp_replace(F.col("nm"), r"[^0-9]", "")).alias("num_key"))
      .write.mode("overwrite").parquet(f"{TMP}/typed"))
    typed = spark.read.parquet(f"{TMP}/typed")

cls = typed.groupBy("name_class").agg(F.count("*").alias("ids"), F.sum("amt").alias("amt")).toPandas()
cls = cls.sort_values("amt", ascending=False)
cls["share_ids"] = cls.ids / cls.ids.sum(); cls["share_dollars"] = cls.amt / cls.amt.sum(); cls["dollars_B"] = cls.amt / 1e9
n_l1 = typed.agg(F.countDistinct("ent_l1")).first()[0]
out(sec(f"§2 ent_l1 and typer v2 ({TIMES['§2 ent_l1 + typer']:.1f} min)",
        tbl(cls[["name_class", "ids", "share_ids", "dollars_B", "share_dollars"]], pct=("share_ids", "share_dollars"),
            dec=("dollars_B",))),
    para(f"{len(typed.columns)} columns typed for {cls.ids.sum():,} ids; {n_l1:,} ent_l1 units; "
         f"{len(FIRST_NAMES):,} first names and {len(BIZ_VOCAB):,} learned business words derived from the data."))

## §3 Entity map v2: `pkg_cpty_entity_map_v2`

- **`ent_l2`:** business ids merge on their cleaned name key. Distinctive keys (a word of ≥ 4 letters used by
  ≤ `DF_MAX` keys, or ≥ `CONC_MIN` ids per FI on average) merge across banks (`N:`, rule `business_name`); generic keys
  merge only within one bank (`F:`, rule `business_name_fi`); digit-only names merge on digits + bank (`D:`, rule
  `numeric_fi`). People, banks and placeholders stay at `ent_l1`.
- **`KEY_IGNORE_SPACES`:** keys that differ only by spaces form one merge key; it is distinctive when any of its
  spellings is distinctive, and its concentration counts all spellings. Its entity id is built from its most common
  spelling, so every key without spelling variants keeps the same `ent_l2` id as before.
- **`business_pair` (v2):** brands named after people. A key that is not distinctive by its words, whose first word is a
  learned first name and whose second word (3+ letters) is not a common surname of P2P names or a generic word (both
  words as written in the key's most common spelling), whose first two words appear in ≤ `BRAND_PAIR_DF_MAX` distinct
  keys, and that has ≥ `BRAND_PAIR_MIN_IDS` ids at ≥ `BRAND_PAIR_MIN_FIS` banks with ≥ `BRAND_PAIR_CONC` ids per bank,
  merges across banks; so do keys starting with a name in `ALWAYS_MERGE_KEYS`. Generic
  pairs (CHECKING CHECKING, HOMEOWNERS ASSOCIATION, CANDLEWOOD SUITES) and common person names (HOA NGUYEN) stay within
  each bank. The largest new merges are listed below for review; `BRAND_PAIR_MERGE = False` turns it off.
- **`ent_l2_fi`:** the same merges, but always within one bank.
- **One row per id:** if an id ever has more than one row, the lowest `ent_l2` is kept and reported below.

In [ ]:
# ---- §3 Entity map v2 ------------------------------------------------------------------------------
assert globals().get("_S0_OK"), "§0 did not finish: fix the problem it shows, then run all cells again"

spark.conf.set("spark.sql.shuffle.partitions", str(2 * SHUFFLE_PARTITIONS))
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "32m")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
KNS = F.regexp_replace(F.col("biz_key"), " ", "") if KEY_IGNORE_SPACES else F.col("biz_key")
ALIAS_NS = sorted({(v.replace(" ", "") if KEY_IGNORE_SPACES else v) for v in KEY_ALIASES.values()})
with timed("§3 entity map"):
    progress("§3 Entity map v2", "1/4 key distinctiveness and brand pairs")
    blocked_words = sorted((set(LEGAL + ORG + GENERIC_KEYS + CHANNEL + NO_MERGE_KEYS) | FIRST_NAMES) - set(SHORT_KEYS_OK))
    BL_DF = F.broadcast(spark.createDataFrame(pd.DataFrame({"biz_key": blocked_words}), "biz_key string")
                         .withColumn("blk_word", F.lit(True)))
    has_key = F.col("biz_key").isNotNull() & (F.col("biz_key") != "")
    kb = typed.where(has_key).select("biz_key", "fi", "type_reason").withColumn("key_ns", KNS)

    # distinctiveness: document frequency of each word over the distinct business keys
    keys = kb.select("biz_key", "key_ns").distinct()
    kdf = (keys.select("biz_key", F.explode(F.array_distinct(F.split("biz_key", " "))).alias("tok"))
               .where(F.col("tok").rlike(r"^[A-Z]{4,}$")))
    tok_df = kdf.groupBy("tok").agg(F.count("*").alias("df"))
    common = set(LEGAL + ORG + CHANNEL + GENERIC_KEYS) | FIRST_NAMES
    key_rare = (kdf.join(tok_df, "tok").where((F.col("df") <= DF_MAX) & ~F.col("tok").isin(sorted(common)))
                   .select("biz_key").distinct())
    rare_ns = keys.join(key_rare, "biz_key").select("key_ns").distinct().withColumn("rare_ns", F.lit(True))
    # v2: rare two-word pairs (brands named after people) and the hand list
    w12 = F.split("biz_key", " ")
    pairs = (keys.where(F.size(w12) >= 2)
                 .select("biz_key", "key_ns", w12.getItem(0).alias("w1"), w12.getItem(1).alias("w2"),
                         F.concat_ws(" ", w12.getItem(0), w12.getItem(1)).alias("pair")))
    pair_df = pairs.groupBy("pair").agg(F.countDistinct("biz_key").alias("pair_df"))
    # a brand named after a person: first name + a second word that is not a common surname or a generic word
    _gen = sorted(set(LEGAL + ORG + GENERIC_KEYS + CHANNEL + STOP + PLACEHOLDERS + NOT_FIRST) | SURNAMES | FIRST_NAMES)
    GEN_DF = F.broadcast(spark.createDataFrame(pd.DataFrame({"w2": _gen}), "w2 string"))
    pairs = (pairs.join(FN_DF.withColumnRenamed("tok", "w1"), "w1", "left_semi")
                  .where((F.length("w1") >= 3) & ~F.col("w1").isin(STOP) & (F.length("w2") >= 3)).join(GEN_DF, "w2", "left_anti"))
    _am = sorted({(k.replace(" ", "") if KEY_IGNORE_SPACES else k) for k in ALWAYS_MERGE_KEYS})
    _am_cond = F.lit(False)                       # hand list: the key starts with a listed name (JOHN DEERE FINANCIAL)
    for _k in _am:
        _am_cond = _am_cond | F.col("key_ns").startswith(_k)
    fi_n = F.coalesce(F.upper(F.trim(F.col("fi"))), F.lit("(none)"))
    per_key = kb.groupBy("biz_key").agg(
        F.count("*").alias("k_ids"), F.countDistinct(fi_n).alias("k_fis"),
        F.avg(F.col("type_reason").isin("merchant", "ach_originator").cast("double")).alias("beh_share"))
    per_ns = kb.groupBy("key_ns").agg(F.count("*").alias("ns_ids"), F.countDistinct(fi_n).alias("ns_fis"))
    # the id of a merged key is the md5 of its most common spelling, so keys without spelling variants keep their ids
    w_rep = Window.partitionBy("key_ns").orderBy(F.desc("k_ids"), "biz_key")
    key_rep = (per_key.join(keys, "biz_key").withColumn("_r", F.row_number().over(w_rep)).filter("_r = 1")
                      .select("key_ns", F.col("biz_key").alias("key_rep")))
    # the pair must come from the key's most common spelling (SKYLINE, MELROSE are not SKY LINE, MEL ROSE)
    pair_ns = (pairs.join(pair_df, "pair").where(F.col("pair_df") <= BRAND_PAIR_DF_MAX)
                    .join(key_rep.withColumnRenamed("key_rep", "biz_key"), ["key_ns", "biz_key"], "left_semi")
                    .select("key_ns").distinct().withColumn("pair_rare", F.lit(bool(BRAND_PAIR_MERGE))))
    key_distinct = (per_key.join(keys, "biz_key").join(per_ns, "key_ns").join(rare_ns, "key_ns", "left")
                           .join(pair_ns, "key_ns", "left").join(key_rep, "key_ns")
                           .withColumn("_old_rule", F.coalesce(F.col("rare_ns"), F.lit(False))
                                                    | (F.col("ns_ids") >= CONC_MIN * F.col("ns_fis"))
                                                    | F.col("key_ns").isin(ALIAS_NS))
                           .withColumn("pair_merge", ~F.col("_old_rule")
                                                     & ((F.coalesce(F.col("pair_rare"), F.lit(False))
                                                         & (F.col("ns_ids") >= BRAND_PAIR_MIN_IDS)
                                                         & (F.col("ns_fis") >= BRAND_PAIR_MIN_FIS)
                                                         & (F.col("ns_ids") >= BRAND_PAIR_CONC * F.col("ns_fis")))
                                                        | _am_cond))
                           .withColumn("distinct_key", F.col("_old_rule") | F.col("pair_merge"))
                           .withColumn("brand_ok", (F.col("k_ids") >= CONC_MIN * F.col("k_fis"))
                                                   & (F.col("beh_share") >= BRAND_BEH_SHARE) & (F.length("biz_key") >= 3))
                           .select("biz_key", "key_ns", "key_rep", "distinct_key", "brand_ok", "pair_merge"))
    key_distinct.write.mode("overwrite").parquet(f"{TMP}/keys")

    progress("§3 Entity map v2", "2/4 merge targets")
    # merge targets (business and numeric ids only; null keys stay out of the key join)
    tsel = [F.col("cpty_id").alias("ent_l1"), F.col("name_class").alias("t_class"),
            F.col("biz_key"), F.col("num_key").alias("t_num"), F.col("fi").alias("t_fi")]
    tgt_biz = (typed.where(has_key).select(*tsel).join(BL_DF, "biz_key", "left")
                    .join(spark.read.parquet(f"{TMP}/keys"), "biz_key", "left"))
    tgt_num = (typed.where(~has_key & F.col("num_key").isNotNull()).select(*tsel)
                    .withColumn("blk_word", F.lit(None).cast("boolean"))
                    .withColumn("key_ns", F.lit(None).cast("string"))
                    .withColumn("key_rep", F.lit(None).cast("string"))
                    .withColumn("distinct_key", F.lit(None).cast("boolean"))
                    .withColumn("brand_ok", F.lit(None).cast("boolean"))
                    .withColumn("pair_merge", F.lit(None).cast("boolean")))
    (tgt_biz.unionByName(tgt_num)
            .withColumn("distinct_key", F.coalesce(F.col("distinct_key"), F.lit(False)))
            .withColumn("brand_ok", F.coalesce(F.col("brand_ok"), F.lit(False)) & ~F.col("biz_key").isin(NO_MERGE_KEYS))
            .withColumn("t_key_ok", F.col("biz_key").isNotNull()
                                    & ((((F.length("biz_key") >= KEY_MIN_LEN) | F.col("biz_key").isin(SHORT_KEYS_OK))
                                        & F.col("blk_word").isNull())
                                       | F.col("brand_ok")))
            .drop("blk_word")
            .write.mode("overwrite").parquet(f"{TMP}/targets"))
    tgt = spark.read.parquet(f"{TMP}/targets")

    progress("§3 Entity map v2", "3/4 ent_l2 for every id")
    fi_s = F.coalesce(F.upper(F.trim(F.col("t_fi"))), F.lit(""))
    emap = (typed.select("cpty_id", "cpty_id_src", "ent_l1", "name_class", "type_reason", "name_clean")
                 .join(tgt, "ent_l1", "left")
                 .withColumn("ent_l0", F.col("cpty_id"))
                 .withColumn("ent_l2", F.when((F.col("t_class") == "business") & F.col("t_key_ok") & F.col("distinct_key"),
                                              F.concat(F.lit("N:"), F.md5("key_rep")))
                                        .when((F.col("t_class") == "business") & F.col("t_key_ok"),
                                              F.concat(F.lit("F:"), F.md5(F.concat_ws("|", "key_rep", fi_s))))
                                        .when((F.col("t_class") == "numeric_name") & (F.length("t_num") >= 4),
                                              F.concat(F.lit("D:"), F.md5(F.concat_ws("|", "t_num", fi_s))))
                                        .otherwise(F.col("ent_l1")))
                 .withColumn("ent_l2_fi", F.when(F.col("ent_l2").startswith("N:"),
                                                 F.concat(F.lit("F:"), F.md5(F.concat_ws("|", "key_rep", fi_s))))
                                           .otherwise(F.col("ent_l2")))
                 .withColumn("rule", F.when(F.col("ent_l2").startswith("N:") & F.coalesce(F.col("pair_merge"), F.lit(False)),
                                            "business_pair")
                                      .when(F.col("ent_l2").startswith("N:"), "business_name")
                                      .when(F.col("ent_l2").startswith("F:"), "business_name_fi")
                                      .when(F.col("ent_l2").startswith("D:"), "numeric_fi")
                                      .when(F.col("ent_l1") != F.col("cpty_id"), "crosswalk_6c_5c")
                                      .otherwise("identity"))
                 .withColumn("key_blocked", F.coalesce((F.col("t_class") == "business") & ~F.col("t_key_ok"), F.lit(False)))
                 .select("cpty_id", "cpty_id_src", "ent_l0", "ent_l1", "ent_l2", "ent_l2_fi", "rule",
                         "name_class", "type_reason", "name_clean", F.col("biz_key").alias("name_key"), "key_blocked"))
    emap.write.mode("overwrite").parquet(f"{TMP}/map_all")
    m_all = spark.read.parquet(f"{TMP}/map_all")
    progress("§3 Entity map v2", "4/4 one row per id, staging")
    # one row per cpty_id: keep the lowest ent_l2 (deterministic) when an id has several rows
    w1 = Window.partitionBy("cpty_id").orderBy("ent_l2", "rule")
    (m_all.withColumn("_r", F.row_number().over(w1)).filter("_r = 1").drop("_r")
          .write.mode("overwrite").parquet(S["cpty_entity_map_v2"]))
    MAP = spark.read.parquet(S["cpty_entity_map_v2"])
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))

progress("§3 Entity map v2", "checks and review tables")
n_dim = DIM.count()
chk = MAP.agg(F.count("*").alias("rows"), F.countDistinct("cpty_id").alias("ids"), F.countDistinct("ent_l1").alias("l1"),
              F.countDistinct("ent_l2").alias("l2"), F.countDistinct("ent_l2_fi").alias("l2fi")).first()
dups = (m_all.groupBy("cpty_id").agg(F.count("*").alias("rows"), F.collect_set("ent_l2").alias("ent_l2_values"))
             .where("rows > 1"))
n_dup = dups.count()
assert chk["rows"] == chk["ids"] == n_dim, f"map v2: {chk['rows']:,} rows, {chk['ids']:,} ids, dim {n_dim:,} ids"
levels = pd.DataFrame([{"level": "raw ids (ent_l0)", "units": chk["ids"]}, {"level": "ent_l1", "units": chk["l1"]},
                       {"level": "ent_l2", "units": chk["l2"]}, {"level": "ent_l2_fi", "units": chk["l2fi"]}])
levels["ids_per_unit"] = chk["ids"] / levels.units
rules = MAP.groupBy("rule").agg(F.count("*").alias("ids"), F.countDistinct("ent_l2").alias("units")).toPandas()
joined = (spark.read.parquet(f"{TMP}/keys").groupBy("key_ns")
               .agg(F.countDistinct("biz_key").alias("spellings"), F.concat_ws(" | ", F.sort_array(F.collect_set("biz_key"))).alias("keys"))
               .where("spellings > 1")
               .join(MAP.where(F.col("rule").isin("business_name", "business_name_fi"))
                        .withColumn("key_ns", F.regexp_replace("name_key", " ", "") if KEY_IGNORE_SPACES else F.col("name_key"))
                        .groupBy("key_ns").agg(F.count("*").alias("ids"), F.countDistinct("ent_l2").alias("units")), "key_ns")
               .orderBy(F.desc("ids")))
n_joined = joined.count()
top_joined = joined.limit(10).toPandas()
pair_new = (MAP.where(F.col("rule") == "business_pair").groupBy("name_key")
               .agg(F.count("*").alias("ids"), F.countDistinct("ent_l2_fi").alias("banks_joined"))
               .orderBy(F.desc("ids")))
n_pair_keys = pair_new.count()
top_pair = pair_new.limit(30).toPandas()
out(sec(f"§3 {TNAME['cpty_entity_map_v2']} (staged, {TIMES['§3 entity map']:.1f} min)", tbl(levels, dec=("ids_per_unit",))),
    sec("Ids and units per rule", tbl(rules.sort_values("ids", ascending=False))),
    para(f"Ids with more than one map row before the reduction: {n_dup:,}" + (" (lowest ent_l2 kept)." if n_dup else ".")),
    sec(f"Keys joined by KEY_IGNORE_SPACES: {n_joined:,} (largest 10)", tbl(top_joined[["keys", "spellings", "ids", "units"]]))
        if KEY_IGNORE_SPACES and n_joined else para("No keys joined by KEY_IGNORE_SPACES."),
    sec(f"New cross-bank merges by business_pair: {n_pair_keys:,} keys (largest 30; banks_joined = within-bank "
        "entities it joins)", tbl(top_pair)) if n_pair_keys else para("No keys merged by business_pair."))

## §4 Canonical bank names: `pkg_fi_routing` and `pkg_fi_name_map`

- **Per routing number:** the canonical name is its dominant name (by dollars) in the latest month it appears;
  spellings are unified only within a routing number; routing numbers carrying the same set of names get the same
  canonical name.
- **Per raw name** (for ids without a routing prefix): the canonical name of the routing numbers it appears under, when
  one canonical name holds ≥ `NAME_MIN_SHARE` of its dollars; otherwise the raw name.
- Before both: `FI_FIXES` (exact names), `FI_FIX_RULES` (patterns), fintech partner names → the partner bank, division
  names → the parent bank, parent names cut short by the feed completed.

Downstream, the canonical bank of a counterparty id is `pkg_fi_routing` via its routing prefix, else `pkg_fi_name_map`
via its raw bank name, else the raw name. Same logic as `pkg_fi_canonical.ipynb`.

In [ ]:
# ---- §4 Canonical bank names -----------------------------------------------------------------------
assert globals().get("_S0_OK"), "§0 did not finish: fix the problem it shows, then run all cells again"

def fi_key(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return None
    k = re.sub(r"[^A-Z0-9]+", " ", str(s).upper()).strip()
    k = re.sub(r"\b([A-Z]) (?=[A-Z]\b)", r"\1", k)
    for w in sorted(FI_STRIP, key=len, reverse=True):
        k = re.sub(rf"\b{w}\b", " ", k)
    k = re.sub(r"\s+", " ", k).strip()
    return k or None
_clean = lambda x: re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9 ]+", " ", str(x).upper())).strip()

with timed("§4 canonical banks"):
    progress("§4 Canonical bank names", "1/3 raw bank names and rule fixes")
    cm = spark.table(T_CPTY_MONTH)
    # every raw FI name with its dollars (for FI_FIX_RULES targets)
    name_amt = (cm.where(F.col("fi").isNotNull()).groupBy("fi")
                  .agg(F.sum(F.col("amount").cast("double")).alias("amt")).toPandas())
    FIXES = dict(FI_FIXES)
    rule_rows = []
    for src_pat, tgt_pat in FI_FIX_RULES:
        cand = name_amt[name_amt.fi.map(lambda x: bool(re.search(tgt_pat, _clean(x))))].sort_values("amt", ascending=False)
        srcs = name_amt[name_amt.fi.map(lambda x: bool(re.search(src_pat, _clean(x))))]
        target = cand.fi.iloc[0] if len(cand) else None
        for s_name, s_amt in zip(srcs.fi, srcs.amt):
            if target is not None and s_name != target:
                FIXES[s_name] = target
            rule_rows.append({"rule": src_pat, "raw name matched": s_name, "dollars_B": s_amt / 1e9,
                              "now named": target if target is not None else "(no target name found: left as is)"})
        if not len(srcs):
            rule_rows.append({"rule": src_pat, "raw name matched": "(none)", "dollars_B": 0.0,
                              "now named": target if target is not None else "(no target name found)"})

    def fi_fix(s):
        # exact fixes and rule fixes, then fintech partner names → the partner bank, division names → the parent
        if s is None or (isinstance(s, float) and np.isnan(s)):
            return s
        s = FIXES.get(s, s)
        m = re.match(FI_PARTNER_RE, _clean(s)) or re.match(FI_DIVISION_RE, _clean(s))
        return m.group(1).strip() if m else s

    def complete_truncated(fixed, known):
        known = sorted(set(known))
        res = {}
        for f in set(fixed) - set(known):
            if f is None or (isinstance(f, float) and np.isnan(f)) or len(str(f)) < 6:
                continue
            hits = [k for k in known if k.startswith(f + "") and k != f]
            if len(hits) == 1:
                res[f] = hits[0]
        return res

    progress("§4 Canonical bank names", "2/3 routing × month × bank name")
    # routing × month × FI name
    rt_raw = (cm.where(F.col("fi").isNotNull())
                .withColumn("routing", F.regexp_extract("cpty_id", r"^(\d{9})-", 1))
                .where(F.col("routing") != "")
                .groupBy("routing", "month", "fi")
                .agg(F.sum(F.col("amount").cast("double")).alias("amt"), F.countDistinct("cpty_id").alias("ids"))
                .toPandas())
    rt_raw["fi_fixed"] = rt_raw.fi.map(fi_fix)
    dim_names = (DIM.where(F.col("fi").isNotNull()).groupBy("fi").agg(F.count("*").alias("ids")).toPandas())
    cm_names = name_amt[["fi"]]
    dim_names = dim_names.merge(cm_names, on="fi", how="outer").fillna({"ids": 0})
    dim_names["ids"] = dim_names.ids.astype("int64")
    dim_names["fi_fixed"] = dim_names.fi.map(fi_fix)
    _known = [str(x).upper() for x in dim_names.fi.dropna()]
    _div_tails = [fi_fix(x) for x in pd.concat([dim_names.fi, rt_raw.fi]).dropna().unique()
                  if re.match(FI_DIVISION_RE, _clean(x))]
    TRUNC_FIX = complete_truncated(_div_tails, _known)
    rt_raw["fi_fixed"] = rt_raw.fi_fixed.replace(TRUNC_FIX)
    dim_names["fi_fixed"] = dim_names.fi_fixed.replace(TRUNC_FIX)
    rt_raw["key"] = rt_raw.fi_fixed.map(fi_key)
    dim_names["key"] = dim_names.fi_fixed.map(fi_key)

    # canonical name per routing number
    rk = (rt_raw.dropna(subset=["key"]).groupby(["routing", "key", "fi_fixed"]).amt.sum().reset_index()
                .sort_values(["routing", "key", "amt"], ascending=[True, True, False]).drop_duplicates(["routing", "key"]))
    RK_SPELL = {(r, k): sp for r, k, sp in zip(rk.routing, rk.key, rk.fi_fixed)}
    last_m = rt_raw.groupby("routing").month.max().rename("last_month")
    first_m = rt_raw.groupby("routing").month.min().rename("first_month")
    latest = (rt_raw.merge(last_m, left_on=["routing", "month"], right_on=["routing", "last_month"])
                    .sort_values(["routing", "amt"], ascending=[True, False]).drop_duplicates("routing")).set_index("routing")
    kspan = rt_raw.groupby(["routing", "key"]).agg(k_first=("month", "min"), k_last=("month", "max"), k_amt=("amt", "sum"))
    rows = []
    for rt, g in rt_raw.groupby("routing"):
        keys_ = g.key.dropna().unique()
        raw = sorted(g.fi.unique())
        lk = latest.loc[rt, "key"]
        canon = RK_SPELL.get((rt, lk), latest.loc[rt, "fi_fixed"])
        if len(raw) == 1:
            ctype, switch = "single name", None
        elif len(keys_) == 1:
            ctype, switch = "spelling variant", None
        elif lk is None or lk not in kspan.loc[rt].index:
            ctype, switch = "concurrent names", None
        else:
            ks = kspan.loc[rt]
            new_first = ks.loc[lk, "k_first"]
            others_last = ks.drop(index=lk).k_last.max()
            ctype = "renamed / acquired" if others_last < new_first else "concurrent names"
            switch = new_first if ctype == "renamed / acquired" else None
        rows.append({"routing": rt, "fi_canonical": canon, "n_names": len(raw), "n_keys": len(keys_),
                     "change_type": ctype, "switch_month": switch, "names_seen": " | ".join(raw[:6]),
                     "first_month": first_m[rt], "last_month": last_m[rt],
                     "ids": int(g.groupby("month").ids.max().max()), "amt": float(g.amt.sum())})
    ROUT = pd.DataFrame(rows)
    ROUT["change_type"] = ROUT.change_type.replace({"renamed / acquired": "multiple names", "concurrent names": "multiple names"})
    kset = rt_raw.dropna(subset=["key"]).groupby("routing").key.apply(lambda s: "|".join(sorted(set(s))))
    ROUT["key_set"] = ROUT.routing.map(kset)
    multi = ROUT[ROUT.change_type == "multiple names"]
    for ks, grp in multi.groupby("key_set"):
        if len(grp) < 2:
            continue
        sub = rt_raw[rt_raw.routing.isin(grp.routing)]
        lm = sub.month.max()
        best = sub[sub.month == lm].groupby("key").amt.sum().idxmax()
        ROUT.loc[grp.index, "fi_canonical"] = sub[sub.key == best].groupby("fi_fixed").amt.sum().idxmax()
    ROUT["kept_raw"] = ROUT.routing.isin(FI_KEEP_RAW_ROUTINGS)
    ROUT.loc[ROUT.kept_raw, "fi_canonical"] = None

    # raw name → canonical name
    rt_c = rt_raw.merge(ROUT.loc[~ROUT.kept_raw, ["routing", "fi_canonical"]], on="routing")
    by = rt_c.groupby(["fi", "fi_canonical"]).agg(amt=("amt", "sum"), routings=("routing", "nunique")).reset_index()
    by["share"] = by.amt / by.groupby("fi").amt.transform("sum").clip(lower=1e-9)
    by = by.sort_values(["fi", "amt"], ascending=[True, False])
    top = by.drop_duplicates("fi")[["fi", "fi_canonical", "share"]].rename(columns={"fi_canonical": "top_canonical"})
    NAMEMAP = dim_names[["fi", "fi_fixed", "key", "ids"]].merge(top, on="fi", how="left")
    clear = NAMEMAP.share >= NAME_MIN_SHARE
    NAMEMAP["source"] = np.select([NAMEMAP.top_canonical.isna(), clear], ["no routing", "routing"], "ambiguous")
    NAMEMAP["fi_canonical"] = np.where(clear, NAMEMAP.top_canonical, NAMEMAP.fi_fixed)
    NAMEMAP["changed"] = NAMEMAP.fi != NAMEMAP.fi_canonical

    progress("§4 Canonical bank names", "3/3 staging")
    # stage both tables (explicit schema; NaN → NULL in text columns)
    def to_sdf(pdf):
        typ = {"object": "string", "bool": "boolean", "int64": "bigint", "int32": "int", "float64": "double"}
        pdf = pdf.copy()
        for c in pdf.columns:
            if pdf[c].dtype == object:
                pdf[c] = pdf[c].astype(object).where(pdf[c].notna(), None)
        schema = ", ".join(f"`{c}` {typ.get(str(pdf[c].dtype), 'string')}" for c in pdf.columns)
        return spark.createDataFrame(pdf, schema)
    rt_out = ROUT.drop(columns=["amt", "switch_month"]).assign(amount=ROUT.amt)
    nm_out = NAMEMAP[["fi", "fi_canonical", "source", "ids"]].rename(columns={"fi": "fi_raw"})
    to_sdf(rt_out).coalesce(1).write.mode("overwrite").parquet(S["fi_routing"])
    to_sdf(nm_out).coalesce(1).write.mode("overwrite").parquet(S["fi_name_map"])

assert ROUT.routing.is_unique and NAMEMAP.fi.is_unique, "bank tables must be one row per key"
ct = ROUT.groupby("change_type").agg(routing_numbers=("routing", "count"), amt=("amt", "sum")).reset_index()
ct["share_dollars"] = ct.amt / ct.amt.sum()
src = NAMEMAP.groupby("source").agg(raw_names=("fi", "count"), ids=("ids", "sum"), changed=("changed", "sum")).reset_index()
canon_of = dict(zip(NAMEMAP.fi, NAMEMAP.fi_canonical))
rr = pd.DataFrame(rule_rows)
if len(rr):
    rr["canonical (raw name)"] = rr["raw name matched"].map(canon_of)
    rr["canonical (target)"] = rr["now named"].map(canon_of)
out(sec(f"§4 {TNAME['fi_routing']}: {len(ROUT):,} routing numbers (staged, {TIMES['§4 canonical banks']:.1f} min)",
        tbl(ct[["change_type", "routing_numbers", "share_dollars"]], pct=("share_dollars",))),
    sec(f"{TNAME['fi_name_map']}: {len(NAMEMAP):,} raw names, {NAMEMAP.fi_canonical.nunique():,} canonical names", tbl(src)),
    sec("FI_FIX_RULES: raw names matched and the name they now take", tbl(rr, dec=("dollars_B",))) if len(rr) else "",
    para(f"FI_FIXES applied: {len(FI_FIXES)} exact, {len(FIXES) - len(FI_FIXES)} from rules. "
         f"Division names completed from truncation: {len(TRUNC_FIX)}."))

## §4b Account-type flags on the map

- **`name_group`:** the first group of `NAME_GROUPS` whose pattern matches the id's cleaned name (PNC own, processors and
  servicers, other named large accounts); empty otherwise.
- **`is_rotator`:** for merged entities (`N:`), each customer's flows with the entity are split by the canonical bank of
  the ids used that month. A customer month is split when it goes through 2 or more banks and the top bank holds less
  than `ROT_TOP_SHARE`; an entity month is split when split customers carry at least `ROT_PAIR_SHARE` of its dollars. An
  entity with at least `ROT_MIN_MONTHS` split months among its last `ROT_LAST` active months is a rotator: its bank moves
  are its own routine, not a sign about the customer, so the metrics leave rotators out of `lost_fi`, `fi_rec_broken`
  and `cpty_bank_moved`. A brand that simply has many ids at many banks (each customer always on the same one) is not,
  and neither is an entity some customer sees through more than `ROT_MAX_BANKS` banks in a month (a merged group of
  separate payers, such as franchisees).
- The flags are added to the staged map; the list below shows the largest rotators and each name group for review.

In [ ]:
# ---- §4b Account-type flags ------------------------------------------------------------------------
assert globals().get("_S0_OK"), "§0 did not finish: fix the problem it shows, then run all cells again"

def name_group_col(c):
    """First NAME_GROUPS group whose pattern matches the (upper-case, cleaned) name, else NULL."""
    e = F.lit(None).cast("string")
    for g, pat in reversed(list(NAME_GROUPS.items())):
        e = F.when(F.coalesce(c.rlike(pat), F.lit(False)), F.lit(g)).otherwise(e)
    return e

RT_DF = F.broadcast(to_sdf(ROUT.loc[~ROUT.kept_raw, ["routing", "fi_canonical"]].rename(columns={"fi_canonical": "fi_rt"})))
NM_DF = F.broadcast(to_sdf(NAMEMAP[["fi", "fi_canonical"]].rename(columns={"fi_canonical": "fi_nm"})))
def canon_fi(df):
    """Adds fi_c (canonical bank) to a frame with cpty_id and raw fi: routing prefix → pkg_fi_routing, else the name map."""
    return (df.withColumn("routing", F.regexp_extract("cpty_id", r"^(\d{9})-", 1))
              .join(RT_DF, "routing", "left").join(NM_DF, "fi", "left")
              .withColumn("fi_c", F.coalesce("fi_rt", "fi_nm", "fi")).drop("routing", "fi_rt", "fi_nm"))

with timed("§4b flags"):
    progress("§4b Account-type flags", "1/2 bank per customer relationship and month (rotators)")
    nids = MAP.where(F.col("ent_l2").startswith("N:")).select("cpty_id", "ent_l2")
    # the id's bank each month: canonical name of its dominant raw bank (by dollars) that month
    idm = (canon_fi(spark.table(T_CPTY_MONTH).where(F.col("fi").isNotNull()).join(nids.select("cpty_id"), "cpty_id")
                         .select("cpty_id", "fi", "month", F.abs(F.col("amount").cast("double")).alias("a")))
             .groupBy("cpty_id", "month").agg(F.max(F.struct("a", "fi_c")).alias("_s"))
             .select("cpty_id", "month", F.col("_s.fi_c").alias("fi_c")))
    em_ = spark.table(T_EDGE_M).where(~F.coalesce(F.col("is_self"), F.lit(False)))
    pr = (em_.where((F.col("dst_kind") == "CUST") & (F.col("src_kind") != "CUST"))
             .select(F.col("dst_id").alias("cust"), F.col("src_id").alias("cpty_id"), "month", "amount")
             .unionByName(em_.where((F.col("src_kind") == "CUST") & (F.col("dst_kind") != "CUST"))
                             .select(F.col("src_id").alias("cust"), F.col("dst_id").alias("cpty_id"), "month", "amount"))
             .join(nids, "cpty_id").join(idm, ["cpty_id", "month"], "left")
             .groupBy("cust", "ent_l2", "month", "fi_c").agg(F.sum(F.abs(F.col("amount").cast("double"))).alias("a"))
             .withColumn("tot", F.sum("a").over(Window.partitionBy("cust", "ent_l2", "month")))
             .groupBy("cust", "ent_l2", "month").agg(F.count("*").alias("n_banks"), F.first("tot").alias("tot"),
                                                    F.max(F.col("a") / F.when(F.col("tot") > 0, F.col("tot"))).alias("top")))
    bm = (pr.withColumn("split", (F.col("n_banks") >= 2) & (F.coalesce(F.col("top"), F.lit(1.0)) < ROT_TOP_SHARE))
            .groupBy("ent_l2", "month").agg(F.sum("tot").alias("tot"), F.max("n_banks").alias("n_banks"),
                                            F.sum(F.when(F.col("split"), F.col("tot")).otherwise(0.0)).alias("split_amt"))
            .withColumn("split_share", F.col("split_amt") / F.when(F.col("tot") > 0, F.col("tot"))))
    wl = Window.partitionBy("ent_l2").orderBy(F.desc("month"))
    rot = (bm.withColumn("_r", F.row_number().over(wl)).where(F.col("_r") <= ROT_LAST)
             .withColumn("split", F.coalesce(F.col("split_share"), F.lit(0.0)) >= ROT_PAIR_SHARE)
             .groupBy("ent_l2").agg(F.sum(F.col("split").cast("int")).alias("rot_split_months"),
                                    F.max("n_banks").alias("rot_max_banks"), F.sum("tot").alias("rot_amt_last"),
                                    F.avg("split_share").alias("rot_split_share"))
             .withColumn("is_rotator", (F.col("rot_split_months") >= ROT_MIN_MONTHS) & (F.col("rot_max_banks") <= ROT_MAX_BANKS)))
    rot.write.mode("overwrite").parquet(f"{TMP}/rotators")
    progress("§4b Account-type flags", "2/2 flags on the staged map")
    (MAP.join(spark.read.parquet(f"{TMP}/rotators").select("ent_l2", "is_rotator", "rot_split_months"), "ent_l2", "left")
        .withColumn("is_rotator", F.coalesce("is_rotator", F.lit(False)))
        .withColumn("rot_split_months", F.coalesce("rot_split_months", F.lit(0)).cast("int"))
        .withColumn("name_group", name_group_col(F.col("name_clean")))
        .write.mode("overwrite").parquet(f"{TMP}/map_flagged"))
    spark.read.parquet(f"{TMP}/map_flagged").write.mode("overwrite").parquet(S["cpty_entity_map_v2"])
    MAP = spark.read.parquet(S["cpty_entity_map_v2"])

assert MAP.count() == n_dim, "§4b: the flagged map must keep one row per cpty_id"
progress("§4b Account-type flags", "review tables")
ROT = spark.read.parquet(f"{TMP}/rotators")
n_rot = ROT.where("is_rotator").count(); n_n = ROT.count()
ent_nm = (MAP.join(DIM.select("cpty_id", F.col("amount").cast("double").alias("amt")), "cpty_id")
             .groupBy("ent_l2").agg(F.max(F.struct("amt", "name_clean")).alias("_s"), F.count("*").alias("ids"))
             .select("ent_l2", F.col("_s.name_clean").alias("name"), "ids"))
top_rot = (ROT.where("is_rotator").join(ent_nm, "ent_l2").orderBy(F.desc("rot_amt_last")).limit(20)
              .select("name", "ids", "rot_split_months", "rot_max_banks", "rot_split_share",
                      (F.col("rot_amt_last") / 1e9).alias("dollars_B_last"))
              .toPandas())
ng = (MAP.join(DIM.select("cpty_id", F.col("amount").cast("double").alias("amt")), "cpty_id")
         .groupBy(F.coalesce("name_group", F.lit("(none)")).alias("name_group"))
         .agg(F.count("*").alias("ids"), F.countDistinct("ent_l2").alias("entities"), (F.sum("amt") / 1e9).alias("dollars_B"),
              F.sum(F.col("is_rotator").cast("long")).alias("rotator_ids"))
         .orderBy("name_group").toPandas())
ng_top = (MAP.where(F.col("name_group").isNotNull()).join(ent_nm.select("ent_l2", F.col("name").alias("ent_name")), "ent_l2")
             .join(DIM.select("cpty_id", F.col("amount").cast("double").alias("amt")), "cpty_id")
             .groupBy("name_group", "ent_name").agg(F.sum("amt").alias("a"), F.count("*").alias("ids"))
             .withColumn("_r", F.row_number().over(Window.partitionBy("name_group").orderBy(F.desc("a"))))
             .where("_r <= 8").orderBy("name_group", "_r")
             .select("name_group", "ent_name", "ids", (F.col("a") / 1e9).alias("dollars_B")).toPandas())
out(sec(f"§4b Account-type flags ({TIMES['§4b flags']:.1f} min)",
        para(f"Rotators: {n_rot:,} of {n_n:,} merged entities with customer flows ({ROT_MIN_MONTHS}+ split months of the "
             f"last {ROT_LAST}; split = customers seeing it through 2+ banks, top bank under {ROT_TOP_SHARE:.0%}, carry "
             f"{ROT_PAIR_SHARE:.0%}+ of its dollars).")),
    sec("Largest rotators (rot_max_banks: most banks one customer used in a month; rot_split_share: share of dollars "
        "with customers using 2+ banks, mean of the last months)", tbl(top_rot, pct=("rot_split_share",), dec=("dollars_B_last",)))
        if len(top_rot) else para("No rotators found."),
    sec("Name groups on the map", tbl(ng, dec=("dollars_B",))),
    sec("Largest entities per name group (check the patterns catch what they should, and nothing else)",
        tbl(ng_top, dec=("dollars_B",))) if len(ng_top) else "")

## §5 Customer dimension: `pkg_cust_dim`

One row per `mdm_id`: the latest `load_dt` from `neo4j_customer` (ties: the row with a `customer_pwr_id`), plus every
`mdm_id` seen as a customer in the edges but missing from `neo4j_customer`. Same logic as customer metrics v3.
v2 adds `is_pnc_own` (the customer name matches the PNC own pattern of `NAME_GROUPS`) and `name_group`.

In [ ]:
# ---- §5 Customer dimension -------------------------------------------------------------------------
assert globals().get("_S0_OK"), "§0 did not finish: fix the problem it shows, then run all cells again"

CUST_COLS = ["mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "party_type",
             "customer_start_dt", "naics_cd", "naics_desc", "load_dt"]
with timed("§5 customer dimension"):
    progress("§5 Customer dimension", "latest row per mdm_id, edge months, flags")
    EM_all = spark.table(T_EDGE_M)
    raw = (spark.table(CUST_TABLE).select(*[nz(c).alias(c) for c in CUST_COLS])
                .filter(F.col("mdm_id").isNotNull()).withColumn("party_type", F.upper("party_type")))
    w = Window.partitionBy("mdm_id").orderBy(F.col("load_dt").desc_nulls_last(), F.col("customer_pwr_id").isNotNull().desc(),
                                             F.col("customer_pwr_id").asc_nulls_last())
    latest = (raw.withColumn("n_load_versions", F.count("*").over(Window.partitionBy("mdm_id")))
                 .withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn"))
    nc = F.upper(F.col("naics_cd"))
    latest = (latest
        .withColumn("naics_status", F.when(nc.isNull(), "missing")
                                     .when(nc.isin(NAICS_PLACEHOLDERS) | ~nc.rlike(r"^[0-9]{2}"), "placeholder").otherwise("valid"))
        .withColumn("naics2", F.when(F.col("naics_status") == "valid", F.substring("naics_cd", 1, 2)))
        .withColumn("is_tm", F.col("customer_pwr_id").isNotNull()))
    fam = latest.filter(F.col("master_pwr_id").isNotNull()).groupBy("master_pwr_id").agg(F.count("*").alias("family_size"))
    shared = (latest.filter(F.col("customer_pwr_id").isNotNull()).groupBy("customer_pwr_id")
                    .agg(F.count("*").alias("_n")).filter("_n > 1").select("customer_pwr_id", F.lit(True).alias("pwr_shared")))
    latest = (latest.join(fam, "master_pwr_id", "left").join(shared, "customer_pwr_id", "left")
                    .withColumn("in_neo4j_customer", F.lit(True)))
    seen = (EM_all.filter(F.col("src_kind") == "CUST").select(F.col("src_id").alias("mdm_id"), "month")
              .unionByName(EM_all.filter(F.col("dst_kind") == "CUST").select(F.col("dst_id").alias("mdm_id"), "month"))
              .groupBy("mdm_id").agg(F.min("month").alias("first_edge_month"), F.max("month").alias("last_edge_month")))
    cdim = (latest.join(seen, "mdm_id", "full")
             .withColumn("in_neo4j_customer", F.coalesce("in_neo4j_customer", F.lit(False)))
             .withColumn("is_tm", F.coalesce("is_tm", F.lit(False)))
             .withColumn("pwr_shared", F.coalesce("pwr_shared", F.lit(False)))
             .withColumn("naics_status", F.coalesce("naics_status", F.lit("missing")))
             .select("mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "party_type",
                     "naics_cd", "naics_desc", "naics2", "naics_status", "is_tm", "pwr_shared", "family_size",
                     "customer_start_dt", "load_dt", "n_load_versions", "in_neo4j_customer",
                     "first_edge_month", "last_edge_month")
             .withColumn("name_group", name_group_col(clean(F.col("customer_name"))))
             .withColumn("is_pnc_own", F.coalesce(F.col("name_group") == "PNC own", F.lit(False))))
    cdim.write.mode("overwrite").parquet(S["cust_dim"])
    CD = spark.read.parquet(S["cust_dim"])

q = CD.agg(F.count("*").alias("mdm_ids"), F.countDistinct("mdm_id").alias("distinct_ids"),
           F.sum(F.col("in_neo4j_customer").cast("long")).alias("in_neo4j_customer"),
           F.sum((~F.col("in_neo4j_customer")).cast("long")).alias("edges_only"),
           F.sum(F.col("first_edge_month").isNotNull().cast("long")).alias("seen_in_edges"),
           F.sum(F.col("is_tm").cast("long")).alias("tm_customers"),
           F.sum(F.col("pwr_shared").cast("long")).alias("pwr_shared_rows"),
           F.countDistinct("master_pwr_id").alias("master_pwr_ids"), F.max("load_dt").alias("latest_load_dt"),
           F.sum(F.col("is_pnc_own").cast("long")).alias("pnc_own"),
           F.sum(F.col("name_group").isNotNull().cast("long")).alias("in_a_name_group")).toPandas()
cng = (CD.where(F.col("name_group").isNotNull()).groupBy("name_group")
         .agg(F.count("*").alias("customers"), F.sum(F.col("is_tm").cast("long")).alias("tm_customers"),
              F.concat_ws(" | ", F.slice(F.sort_array(F.collect_set("customer_name")), 1, 6)).alias("examples"))
         .orderBy("name_group").toPandas())
assert int(q.mdm_ids[0]) == int(q.distinct_ids[0]), "pkg_cust_dim must have one row per mdm_id"
out(sec(f"§5 {TNAME['cust_dim']} (staged, {TIMES['§5 customer dimension']:.1f} min)", tbl(q.drop(columns=["distinct_ids"]))),
    sec("Customers per name group", tbl(cng)) if len(cng) else para("No customer matches a name group."))

## §5b Replacement map: `pkg_cpty_replacement`

For every TM customer and direction, outside counterparties (no self payments) are taken at entity level (`ent_l2`) per
month. Then:
- **Stop:** a steady entity (paid in at least `REPL_STEADY_MIN` of the 6 months up to its last payment) stops paying for
  at least `REPL_SILENT` months. Its level is the median of its last 3 payments.
- **Start:** an entity new to the customer (not seen in the `REPL_NEW_GAP` months before; in the first `REPL_NEW_GAP`
  months of history from month `REPL_NEW_HIST` on). Its level is the mean of its first 2 calendar months.
- **Candidate pair:** the start falls within `REPL_WINDOW` month of the month after the stop; the levels are within
  `REPL_AMT_TOL`; and the two share a name word of 4+ letters that is not generic or a first name (`name`; `last_name`
  when both are people), or the same canonical bank (`bank`).
- **Busy months:** a customer month with at least `REPL_BUSY_MIN` new counterparties and more than the customer's own p90
  needs a business-name or bank link; a last-name link alone is not enough there.
- **One to one:** each stop and each start keeps its best-scored partner (link, then amount closeness, then timing), and
  only pairs that are each other's best are kept.
- **Left out:** customer-months with more than `REPL_MAX_PER_MONTH` steady stops or new counterparties (collectors and
  processors with thousands of same-amount payers).

Used downstream: `lost_cpty` and `rec_broken` net of replacements, `cpty_replaced_{dir}`, continuation in periodicity,
contagion. One row per replaced stop.

In [ ]:
# ---- §5b Replacement map ---------------------------------------------------------------------------
assert globals().get("_S0_OK"), "§0 did not finish: fix the problem it shows, then run all cells again"

with timed("§5b replacement map"):
    progress("§5b Replacement map", "1/4 monthly series of TM customers' outside counterparties")
    LAST_I = len(MONTHS) - 1
    MI_DF = F.broadcast(spark.createDataFrame(pd.DataFrame({"month": MONTHS, "mi": list(range(len(MONTHS)))}),
                                              "month string, mi bigint"))
    em = spark.table(T_EDGE_M).where(~F.coalesce(F.col("is_self"), F.lit(False)))
    a = F.col("amount").cast("double")
    rows = (em.where((F.col("dst_kind") == "CUST") & (F.col("src_kind") != "CUST"))
              .select(F.col("dst_id").alias("cust"), F.lit("in").alias("dir"), F.col("src_id").alias("cpty_id"), "month",
                      a.alias("a"))
              .unionByName(em.where((F.col("src_kind") == "CUST") & (F.col("dst_kind") != "CUST"))
              .select(F.col("src_id").alias("cust"), F.lit("out").alias("dir"), F.col("dst_id").alias("cpty_id"), "month",
                      a.alias("a"))))
    TMC = CD.where("is_tm").select(F.col("mdm_id").alias("cust"))
    ser = (rows.join(TMC, "cust").join(MAP.select("cpty_id", F.col("ent_l2").alias("ent")), "cpty_id", "left")
               .withColumn("ent", F.coalesce("ent", "cpty_id"))
               .groupBy("cust", "dir", "ent", "month").agg(F.sum("a").alias("a"))
               .where(F.col("a") > 0).join(MI_DF, "month").drop("month"))
    ser.write.mode("overwrite").parquet(f"{TMP}/repl_series")
    ser = spark.read.parquet(f"{TMP}/repl_series")

    progress("§5b Replacement map", "2/4 stops, starts and candidate pairs")
    k = Window.partitionBy("cust", "dir", "ent").orderBy("mi")
    a1, a2 = F.lag("a", 1).over(k), F.lag("a", 2).over(k)
    x = (ser.withColumn("prev_mi", F.lag("mi").over(k)).withColumn("next_mi", F.lead("mi").over(k))
            .withColumn("n6", F.count("*").over(k.rangeBetween(-5, 0)))
            # median of the last 3 paid months = sum − max − min
            .withColumn("med3", F.col("a") + a1 + a2 - F.greatest("a", a1, a2) - F.least("a", a1, a2))
            .withColumn("a_next", F.when(F.lead("mi").over(k) == F.col("mi") + 1, F.lead("a").over(k)).otherwise(F.lit(0.0))))
    is_stop = ((F.col("n6") >= REPL_STEADY_MIN)
               & ((F.col("next_mi").isNull() & (F.col("mi") + REPL_SILENT <= LAST_I)) | (F.col("next_mi") > F.col("mi") + REPL_SILENT)))
    is_start = (F.col("prev_mi").isNull() & (F.col("mi") >= REPL_NEW_HIST)) | (F.col("prev_mi") < F.col("mi") - REPL_NEW_GAP)
    (x.where(is_stop | is_start).select("cust", "dir", "ent", "mi", "a", "a_next", "med3",
                                         is_stop.alias("is_stop"), is_start.alias("is_start"))
      .write.mode("overwrite").parquet(f"{TMP}/repl_events"))
    ev = spark.read.parquet(f"{TMP}/repl_events")
    # per customer-month: new counterparties (busy rule) and the cap on stops and starts
    cm_n = (ev.groupBy("cust", "dir", "mi").agg(F.sum(F.col("is_start").cast("int")).alias("n_new"),
                                               F.sum(F.col("is_stop").cast("int")).alias("n_stop")))
    cm_n.write.mode("overwrite").parquet(f"{TMP}/repl_cm"); cm_n = spark.read.parquet(f"{TMP}/repl_cm")
    p90 = (ser.select("cust", "dir", "mi").distinct().join(cm_n, ["cust", "dir", "mi"], "left").fillna(0, ["n_new"])
              .groupBy("cust", "dir").agg(F.expr("percentile_approx(n_new, 0.9)").alias("p90")))
    busy = (cm_n.join(p90, ["cust", "dir"])
                .select("cust", "dir", F.col("mi").alias("f_mi"),
                        ((F.col("n_new") >= REPL_BUSY_MIN) & (F.col("n_new") > F.col("p90"))).alias("busy")))
    over = cm_n.where((F.col("n_new") > REPL_MAX_PER_MONTH) | (F.col("n_stop") > REPL_MAX_PER_MONTH)).select("cust", "dir", "mi")
    n_over = over.count()
    ev_ok = ev.join(over, ["cust", "dir", "mi"], "left_anti")
    stop = ev_ok.where("is_stop").select("cust", "dir", F.col("ent").alias("old_ent"), F.col("mi").alias("s_mi"),
                                         F.col("med3").alias("old_amt"))
    start = ev_ok.where("is_start").select("cust", "dir", F.col("ent").alias("new_ent"), F.col("mi").alias("f_mi"),
                                           ((F.col("a") + F.col("a_next")) / F.when(F.col("mi") < LAST_I, 2.0).otherwise(1.0)).alias("new_amt"))
    # candidate pairs: each stop is spread over the start months it accepts and the amount buckets next to its own
    # (log buckets of width REPL_AMT_TOL), so the join is an equi-join on customer × direction × month × bucket; a join
    # on customer × direction alone multiplies every stop by every start of a large customer
    import math
    LB = math.log1p(REPL_AMT_TOL)
    bkt = lambda c: F.floor(F.log(c) / F.lit(LB)).cast("long")
    stop_x = (stop.where(F.col("old_amt") > 0)
                  .withColumn("f_mi", F.explode(F.sequence(F.col("s_mi") + 1 - REPL_WINDOW, F.col("s_mi") + 1 + REPL_WINDOW)))
                  .withColumn("bk", F.explode(F.sequence(bkt(F.col("old_amt")) - 1, bkt(F.col("old_amt")) + 1))))
    start_x = start.where(F.col("new_amt") > 0).withColumn("bk", bkt(F.col("new_amt")))
    cand = (stop_x.join(start_x, ["cust", "dir", "f_mi", "bk"]).drop("bk")
                .where((F.col("new_ent") != F.col("old_ent"))
                       & (F.abs(F.col("new_amt") / F.col("old_amt") - 1) <= REPL_AMT_TOL))
                .join(busy, ["cust", "dir", "f_mi"], "left")
                .withColumn("busy", F.coalesce("busy", F.lit(False)))
                .withColumn("pid", F.monotonically_increasing_id()))
    cand.write.mode("overwrite").parquet(f"{TMP}/repl_cand")
    cand = spark.read.parquet(f"{TMP}/repl_cand")

    progress("§5b Replacement map", "3/4 names, banks and links")
    # entity attributes: dominant name, class and canonical bank (by lifetime dollars)
    ents = cand.select(F.col("old_ent").alias("ent")).unionByName(cand.select(F.col("new_ent").alias("ent"))).distinct()
    attr = (canon_fi(MAP.select("cpty_id", F.col("ent_l2").alias("ent"), "name_class", "name_clean")
                        .join(ents, "ent").join(DIM.select("cpty_id", "fi", F.col("amount").cast("double").alias("amt")), "cpty_id"))
              .groupBy("ent").agg(F.max(F.struct("amt", "name_clean", "name_class", "fi_c")).alias("_s"))
              .select("ent", F.col("_s.name_clean").alias("nm"), F.col("_s.name_class").alias("cls"), F.col("_s.fi_c").alias("bank")))
    side = lambda p: attr.select(F.col("ent").alias(f"{p}_ent"), F.col("nm").alias(f"{p}_name"),
                                 F.col("cls").alias(f"{p}_class"), F.col("bank").alias(f"{p}_bank"))
    c2 = cand.join(side("old"), "old_ent", "left").join(side("new"), "new_ent", "left")
    # name link: a shared word of 4+ letters that is not generic and not a first name
    blk = sorted(set(LEGAL + ORG + GENERIC_KEYS + CHANNEL + STOP + PERSON_EXTRA + PLACEHOLDERS) | FIRST_NAMES)
    BLK = F.broadcast(spark.createDataFrame(pd.DataFrame({"w": blk}), "w string"))
    words = lambda p: (c2.select("pid", F.explode(F.array_distinct(F.split(F.coalesce(F.col(f"{p}_name"), F.lit("")), " "))).alias("w"))
                         .where(F.col("w").rlike(r"^[A-Z]{4,}$")).join(BLK, "w", "left_anti"))
    nl = words("old").join(words("new"), ["pid", "w"]).groupBy("pid").agg(F.first("w").alias("link_word"))
    W_LINK = {"name": 3.0, "last_name": 2.0, "bank": 1.0}
    both_person = (F.col("old_class") == "person") & (F.col("new_class") == "person")
    c3 = (c2.join(nl, "pid", "left")
            .withColumn("link", F.when(F.col("link_word").isNotNull() & both_person, "last_name")
                                 .when(F.col("link_word").isNotNull(), "name")
                                 .when(F.col("old_bank").isNotNull() & (F.col("old_bank") == F.col("new_bank")), "bank"))
            .where(F.col("link").isNotNull() & (~F.col("busy") | (F.col("link") != "last_name")))
            .withColumn("amt_ratio", F.col("new_amt") / F.col("old_amt"))
            .withColumn("score", F.element_at(F.create_map(*[F.lit(v) for kv in W_LINK.items() for v in kv]), F.col("link"))
                                 + (1 - F.abs(F.col("amt_ratio") - 1) / REPL_AMT_TOL)
                                 + F.when(F.col("f_mi") == F.col("s_mi") + 1, 0.5).otherwise(0.0)))
    ro = Window.partitionBy("cust", "dir", "old_ent", "s_mi").orderBy(F.desc("score"), "new_ent", "f_mi")
    rn = Window.partitionBy("cust", "dir", "new_ent", "f_mi").orderBy(F.desc("score"), "old_ent", "s_mi")
    mo = lambda c, n: MI_DF.select(F.col("mi").alias(c), F.col("month").alias(n))
    repl = (c3.withColumn("_ro", F.row_number().over(ro)).withColumn("_rn", F.row_number().over(rn))
              .where((F.col("_ro") == 1) & (F.col("_rn") == 1))
              .join(mo("s_mi", "old_last_month"), "s_mi").join(mo("f_mi", "new_first_month"), "f_mi")
              .select(F.col("cust").alias("mdm_id"), F.col("dir").alias("direction"),
                      "old_ent", "old_name", "old_class", "old_bank", "old_last_month", "old_amt",
                      "new_ent", "new_name", "new_class", "new_bank", "new_first_month", "new_amt",
                      "amt_ratio", "link", "link_word", "busy", "score"))
    progress("§5b Replacement map", "4/4 one-to-one matching, staging")
    repl.write.mode("overwrite").parquet(S["cpty_replacement"])
    RP = spark.read.parquet(S["cpty_replacement"])

progress("§5b Replacement map", "checks and review tables")
n_stop = stop.count(); n_cand = cand.count(); n_rp = RP.count()
dup = RP.groupBy("mdm_id", "direction", "old_ent", "old_last_month").count().where("count > 1").count() \
    + RP.groupBy("mdm_id", "direction", "new_ent", "new_first_month").count().where("count > 1").count()
assert dup == 0, "§5b: the replacement map must be one to one"
by = RP.groupBy("direction", "link").agg(F.count("*").alias("pairs"), F.sum("old_amt").alias("old_amt"),
                                          F.sum(F.col("busy").cast("long")).alias("in_busy_months")).toPandas()
stops_by = stop.groupBy("dir").agg(F.count("*").alias("steady_stops"), F.sum("old_amt").alias("stop_amt")).toPandas()
rp_by = RP.groupBy(F.col("direction").alias("dir")).agg(F.count("*").alias("replaced"),
                                                        F.sum("old_amt").alias("replaced_amt")).toPandas()
cov = stops_by.merge(rp_by, on="dir", how="left").fillna(0)
cov["share_stops_replaced"] = cov.replaced / cov.steady_stops.clip(lower=1)
cov["share_dollars_replaced"] = cov.replaced_amt / cov.stop_amt.clip(lower=1)
ex = (RP.orderBy(F.desc("old_amt")).limit(15)
        .select("direction", "old_name", "new_name", "old_last_month", "new_first_month", "old_amt", "amt_ratio", "link", "link_word")
        .toPandas())
out(sec(f"§5b {TNAME['cpty_replacement']}: {n_rp:,} replacements (staged, {TIMES['§5b replacement map']:.1f} min)",
        para(f"{n_stop:,} steady stops at TM customers; {n_cand:,} candidate pairs within the window and amount tolerance; "
             f"{n_rp:,} kept after the link rule and one-to-one matching. {n_over:,} customer-months with more than "
             f"{REPL_MAX_PER_MONTH:,} stops or new counterparties were left out.")),
    sec("Steady stops replaced, by direction (old_amt = monthly level of the stopped counterparty)",
        tbl(cov[["dir", "steady_stops", "replaced", "share_stops_replaced", "share_dollars_replaced"]],
            pct=("share_stops_replaced", "share_dollars_replaced"))),
    sec("Replacements by link", tbl(by.sort_values(["direction", "pairs"], ascending=[True, False]))) if len(by) else "",
    sec("Largest replacements (check that they read as the same payer under a new name or bank)",
        tbl(ex, dec=("amt_ratio",))) if len(ex) else "")

## §6 Publish

Every staged table replaces the current one together: the current folder is set aside, the staged folder is moved into
place, and the table definition is re-created as an external table. Then the temporary folders are removed. Run the
printed statements in Impala.

In [ ]:
# ---- §6 Publish ------------------------------------------------------------------------------------
assert globals().get("_S0_OK"), "§0 did not finish: fix the problem it shows, then run all cells again"

with timed("§6 publish"):
    actions = []
    for k in OUT_KEYS:
        progress("§6 Publish", f"{T[k]} ({OUT_KEYS.index(k) + 1} of {len(OUT_KEYS)})")
        t, final, staged = T[k], P[k], S[k]
        assert exists(f"{staged}/_SUCCESS"), f"{k} was not staged"
        old_loc = None
        if table_exists(t):
            old_loc = table_location(t)
            kind = [r[1] for r in spark.sql(f"DESCRIBE TABLE EXTENDED {t}").collect() if r[0] == "Type"][0].strip().upper()
            if kind == "EXTERNAL":
                spark.sql(f"ALTER TABLE {t} SET TBLPROPERTIES ('external.table.purge' = 'false')")
            spark.sql(f"DROP TABLE {t}")              # external: definition only; managed: its files too
        trash = f"{final}__old"
        delete(trash)
        if exists(final):
            rename(final, trash)
        rename(staged, final)
        cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in spark.read.parquet(final).schema.fields)
        spark.sql(f"CREATE EXTERNAL TABLE {t} ({cols}) STORED AS PARQUET LOCATION '{final}' "
                  "TBLPROPERTIES ('external.table.purge' = 'false')")
        spark.catalog.refreshTable(t)
        delete(trash)
        n = spark.table(t).count()
        actions.append({"table": t, "rows": n, "GB": size_gb(final),
                        "note": "" if old_loc in (None, final) else f"previous location {old_loc} left as is"})
    delete(STAGE); delete(TMP)

pub = pd.DataFrame(actions)
times = pd.DataFrame([{"step": k, "minutes": v} for k, v in TIMES.items()])
out(sec("§6 Published", tbl(pub, dec=("GB",))),
    sec(f"Time: {times.minutes.sum():.1f} min", tbl(times, dec=("minutes",))),
    para("Impala: " + " ".join(f"INVALIDATE METADATA {T[k]};" for k in OUT_KEYS)))